<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">


<h1 style="text-align:center; color:#003366; font-size:48px;">
PROYECTO FINAL
</h1>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">


<div style="text-align:center;">

<h2 style="color:#003366;">Project Conventions:</h2>

<p style="color:#777777;">
All code and comments are written in English.<br>
However, database fields preserve their original language:<br>
- Global datasets (from international sources) are in English<br>
</p>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Descriptive Statistics
</h2>

<h2 style="text-align:center; color:green; font-size:18px;">
In this script, the analysis begins from a global perspective and then moves into a breakdown by Income Groups.<br>
After that, we examine the global energy model—focusing on renewable and fossil systems across countries—followed<br>
by an assessment of how nuclear power can support the global transition.
Finally, we compare the global electricity grid with total primary energy consumption.
</h2>

In [189]:
# =====================================
# STANDARD LIBRARY IMPORTS
# =====================================
import json
import os
import copy

import requests

# =====================================
# ENVIRONMENT VARIABLES
# =====================================
from dotenv import load_dotenv

load_dotenv()
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN")

# =====================================
# DATA ANALYSIS & MATHEMATICS
# =====================================
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =====================================
# DATA VISUALIZATION (MATPLOTLIB / SEABORN)
# =====================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# =====================================
# INTERACTIVE VISUALIZATION (PLOTLY)
# =====================================
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# =====================================
# MAPPING & GEO VISUALIZATION
# =====================================
import folium
import branca.colormap as cm
import pydeck as pdk

# =====================================
# MACHINE LEARNING MODELS
# =====================================
from sklearn.linear_model import LinearRegression

# =====================================
# UTILITIES
# =====================================
import pycountry


In [190]:
# Import OWID tables (2000–2024), fully completed and ready for global analysis
Country_Owid_2000_2024 = pd.read_csv(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Country_Owid_2000_2024.csv")
Country_Global_2000_2024 = pd.read_csv(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Country_Global_2000_2024.csv")
# Create a working copy (good practice)
owid_completed = Country_Owid_2000_2024.copy()
owid_global = Country_Global_2000_2024.copy()

<hr style="border:0;height:2px;background:#777;width:65%;margin:20px auto;box-shadow:0 2px 6px rgba(0,0,0,0.25);">

<h2 style="text-align:center;color:gold;font-size:26px;margin:10px 0;">
Quick Summary of Global Energy Metrics (OWID)
</h2>

<h3 style="text-align:center;color:green;font-size:18px;margin:8px 0;">
<a href="https://ourworldindata.org/grapher/world-bank-income-groups">income_groups</a> |
<a href="https://github.com/owid/energy-data/blob/master/owid-energy-codebook.csv">owid_codebook</a> |
<a href="https://ourworldindata.org/energy-substitution-method">substitution method</a>
</h3>

<h4 style="text-align:center;color:#555;font-size:15px;line-height:1.4;margin:10px auto;width:75%;">
The substitution method adjusts differences between fossil fuels (measured as primary energy, including losses) and clean sources (measured as electricity output).<br>
It estimates how much fossil energy would be needed to generate the same electricity, enabling a fair comparison across sources.<br><br>

<strong>Key concept:</strong> convert clean electricity into its fossil equivalent using a thermal efficiency factor.<br>
<strong>Result:</strong> consistent units (TWh), comparable shares, and a coherent total energy mix.
</h4>

<h3 style="text-align:center;color:red;font-size:17px;margin:6px 0;">
OWID highlights: category sums ≠ total primary energy
</h3>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Evolution of global energy consumption (2000–2024)
</h2>

<h2 style="text-align:center; color:grey; font-size:18px;">
Let’s begin with a global overview of how energy production has evolved,<br>
to get a clear sense of the worldwide trajectory from 2000 to 2024
</h2>
<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:22px;">
    Energy Consumption Metrics
</h2>

<h3 style="text-align:center; color:grey; font-size:18px; line-height:1.5;">
    These indicators measure total national energy consumption across different sources.<br>
    Because they are expressed in absolute terms, they allow us to compare the scale of energy use<br>
    between countries, technologies, and fuel types without population adjustments.<br> 
    All values are expressed in:
</h3>

<h2 style="text-align:center; color:green; font-size:22px; margin-top:5px;">
    terawatt‑hours (TWh)
</h2>

<h3 style="text-align:center; color:grey; font-size:18px; margin-top:0;">
    which makes direct comparison across energy sources straightforward.
</h3>

In [191]:
# Filter the dataset: keep only World data from the year 2000 onward

global_cons = owid_global[owid_global["country"] == "World"]

# Agregar consumo energético global por año
global_energy = (
    global_cons[
        ["year",
        "renewables_consumption",
        "other_renewable_consumption",
        "biofuel_consumption",
        "fossil_fuel_consumption",
        "nuclear_consumption",
        "low_carbon_consumption",
        "primary_energy_consumption"]].groupby("year").sum().round(2))

# Mostrar primeras filas
global_energy.head(30)


,renewables_consumption,other_renewable_consumption,biofuel_consumption,fossil_fuel_consumption,nuclear_consumption,low_carbon_consumption,primary_energy_consumption
year,,,,,,,
2000,8152.47,577.88,132.51,94467.26,7168.52,15320.99,110416.40
2001,7964.84,600.24,131.42,95595.23,7330.21,15295.05,111493.29
2002,8138.82,646.16,149.31,97731.23,7386.15,15524.97,113894.43
2003,8177.06,682.60,168.93,101858.21,7197.06,15374.12,117849.60
2004,8811.90,737.76,201.20,106903.77,7483.36,16295.26,123841.74
2005,9151.29,798.36,234.48,110657.06,7442.49,16593.78,127936.72
2006,9598.00,850.22,294.60,113695.70,7494.95,17092.95,131481.20
2007,9962.66,922.82,391.65,117667.51,7303.11,17265.77,135617.98
2008,10720.29,987.29,535.94,118540.05,7222.83,17943.13,137229.41


In [192]:
fig = go.Figure()

# 1. Fossil fuels (bottom layer)
fig.add_trace(go.Scatter(
    x=global_cons["year"],
    y=global_cons["fossil_fuel_consumption"],
    name="Fossil fuels",
    mode="lines",
    line=dict(width=0.5, color="orange"),
    stackgroup="one"
))

# 2. Renewables (middle layer)
fig.add_trace(go.Scatter(
    x=global_cons["year"],
    y=global_cons["renewables_consumption"],
    name="Renewables",
    mode="lines",
    line=dict(width=0.5, color="green"),
    stackgroup="one"
))

# 3. Nuclear (top layer)
fig.add_trace(go.Scatter(
    x=global_cons["year"],
    y=global_cons["nuclear_consumption"],
    name="Nuclear",
    mode="lines",
    line=dict(width=0.5, color="royalblue"),
    stackgroup="one"
))

# 4. Low‑carbon line
fig.add_trace(go.Scatter(
    x=global_cons["year"],
    y=global_cons["low_carbon_consumption"],
    name="Low‑carbon",
    mode="lines",
    line=dict(width=2, color="black", dash="dash")
))

# --- UNIFIED PREMIUM LAYOUT ---
fig.update_layout(
    title=dict(
        text="Global Energy Consumption by Source",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838")
    ),
    xaxis_title="Year",
    yaxis_title="Energy consumption (TWh)",

    # Background
    plot_bgcolor= "#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=580,
    template="plotly_white",

    # Global text style
    font=dict(color="#383838", size=14),

    # Axes
    xaxis=dict(
        showgrid=False,
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))
    ),
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))
    ),

    # Legend
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="right",
        x=1,
        font=dict(color="#383838", size=14)))

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
# Example: save inside Quarto_Report/images/
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\00_global_energy_stacked_area.html")
# Show the figure
fig.show()


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px; margin-bottom: 20px;">

<span style="font-size:22px; font-weight:700;">
Global Energy Consumption by Source: An Upward Trajectory<br>
</span>

<br>

<span style="font-size:18px;">
This stacked area chart illustrates the absolute growth in global energy demand since the year 2000, measured in Terawatt-hours (TWh).<br>
While low-carbon sources like renewables and nuclear (the green and blue bands) have grown significantly over the last two decades, this growth has primarily served to meet <i>new</i> energy demand rather than replacing existing fossil fuel use.<br>
Consequently, despite the clean energy boom, total fossil fuel consumption (the large orange area) continues to rise in absolute terms.
</span>

</div>


In [193]:
# 1. Filter the dataset for the world from the year 2000 onward
df_world_shares = owid_global[owid_global["country"] == "World"]

# 2. Select the energy share columns and set the year as the index
df_shares = df_world_shares.set_index("year")[[
    "biofuel_share_energy",
    "coal_share_energy",
    "electricity_share_energy",
    "fossil_share_energy",
    "gas_share_energy",
    "hydro_share_energy",
    "low_carbon_share_energy",
    "nuclear_share_energy",
    "oil_share_energy",
    "other_renewables_share_energy",
    "renewables_share_energy",
    "solar_share_energy",
    "wind_share_energy",]].round(2)

df_shares["total_share"] = df_shares.sum(axis=1)

df_shares.head(30)

,biofuel_share_energy,coal_share_energy,electricity_share_energy,fossil_share_energy,gas_share_energy,hydro_share_energy,low_carbon_share_energy,nuclear_share_energy,oil_share_energy,other_renewables_share_energy,renewables_share_energy,solar_share_energy,wind_share_energy,total_share
year,,,,,,,,,,,,,,
2000,0.12,25.01,15.28,86.04,21.86,6.70,13.96,6.53,39.18,0.53,7.43,0.00,0.08,222.72
2001,0.12,25.14,15.33,86.21,21.93,6.42,13.79,6.61,39.14,0.54,7.18,0.00,0.10,222.51
2002,0.13,25.59,15.52,86.29,22.12,6.35,13.71,6.52,38.58,0.57,7.19,0.00,0.13,222.70
2003,0.14,26.89,15.46,86.89,21.95,6.10,13.11,6.14,38.05,0.58,6.98,0.01,0.15,222.45
2004,0.16,27.36,15.42,86.77,21.74,6.20,13.23,6.07,37.67,0.60,7.15,0.01,0.19,222.57
2005,0.18,28.45,15.51,86.96,21.56,6.15,13.04,5.85,36.95,0.63,7.19,0.01,0.22,222.70
2006,0.23,29.12,15.67,86.93,21.54,6.18,13.07,5.73,36.27,0.65,7.34,0.01,0.27,223.01
2007,0.29,29.82,15.86,87.20,21.74,6.06,12.80,5.41,35.64,0.68,7.38,0.02,0.34,223.24
2008,0.39,29.89,16.01,86.85,22.02,6.29,13.15,5.29,34.94,0.72,7.85,0.02,0.43,223.85


In [194]:
# Ensure correct X-axis
x_data = df_shares["year"] if "year" in df_shares.columns else df_shares.index

fig = go.Figure()

# 1. Fossil fuels
fig.add_trace(go.Bar(
    x=x_data,
    y=df_shares["fossil_share_energy"],
    name="Fossil fuels",
    marker_color="orange"     # EXACT same as area chart
))

# 2. Renewables
fig.add_trace(go.Bar(
    x=x_data,
    y=df_shares["renewables_share_energy"],
    name="Renewables",
    marker_color="green"      # EXACT same as area chart
))

# 3. Nuclear
fig.add_trace(go.Bar(
    x=x_data,
    y=df_shares["nuclear_share_energy"],
    name="Nuclear",
    marker_color="royalblue"  # EXACT same as area chart
))

# --- UNIFIED PREMIUM LAYOUT ---
fig.update_layout(
    barmode='stack',
    title=dict(
        text="Global Energy Mix Share",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838")
    ),
    xaxis_title="Year",
    yaxis_title="Percentage of Primary Energy Consumption (%)",

    # Background
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=580,
    template="plotly_white",

    # Global text style
    font=dict(color="#383838", size=14),

    # Axes
    xaxis=dict(
        showgrid=False,
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))
    ),
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16)),
        range=[0, 100]
    ),

    # Legend
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="right",
        x=1,
        font=dict(color="#383838", size=14)
    )
)

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
# Example: save inside Quarto_Report/images/
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\01_global_energy_shares.html")
# Show the figure
fig.show()

<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
Global Energy Mix Share: The Slow Pace of the Transition<br>
</span>

<br>

<span style="font-size:18px;">
This 100% stacked bar chart provides a proportional view of the global energy mix, highlighting the persistent dominance of fossil fuels.<br>
As the orange bars show, fossil fuels have consistently accounted for over 80% of total primary energy consumption throughout the 21st century.<br>
Although the relative share of renewables (green) is slowly expanding and gradually eating into the fossil fuel percentage, the overall shift in the energy mix remains sluggish compared to the pace required to meet global climate targets.
</span>

</div>

In [195]:
# 1. Select the data
df_fossil = df_shares[[
    "coal_share_energy",
    "oil_share_energy",
    "gas_share_energy",
]]

# Ensure the X-axis is correct
x_data = df_fossil.index if "year" not in df_fossil.columns else df_fossil["year"]

# 2. Create the area chart
fig_fossil = px.area(
    df_fossil,
    x=x_data,
    y=df_fossil.columns,
    color_discrete_map={
        "coal_share_energy": "#4c4c4c",   # coal (dark gray)
        "oil_share_energy": "#b5651d",    # oil (brown)
        "gas_share_energy": "#C39BD3"     # gas (light purple)
    }
)

# --- PRO TIP: Clean variable names for legend and tooltip ---
clean_names = {
    'coal_share_energy': 'Coal', 
    'oil_share_energy': 'Oil', 
    'gas_share_energy': 'Gas'
}
fig_fossil.for_each_trace(lambda t: t.update(
    name=clean_names[t.name],
    legendgroup=clean_names[t.name],
    hovertemplate=t.hovertemplate.replace(t.name, clean_names[t.name])
))

# --- FIRST LAYOUT BLOCK ---
fig_fossil.update_layout(
    title="Global Fossil Energy Shares (Coal, Oil, Gas)",
    xaxis_title="Year",
    yaxis_title="Share (%)",
    
    # Background and theme
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=580,
    template="plotly_white",

    # Global text color
    font=dict(color="#383838"),

    # Unified hover mode
    hovermode="x unified",

    # Axis customization
    xaxis=dict(
        showgrid=False,
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838"))
    ),
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838"))
    ),
    
    # Horizontal legend at the top
    legend=dict(
        title=None, 
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="right",
        x=1,
        font=dict(color="#383838")
    )
)

# --- UPDATED LAYOUT BLOCK ---
fig_fossil.update_layout(
    
    # 1. Centered and enlarged title
    title=dict(
        text="Global Fossil Energy Shares",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    
    xaxis_title="Year",
    yaxis_title="Share (%)",
    
    # Background and theme
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=580,
    template="plotly_white",

    # 2. Global text size and color
    font=dict(
        color="#383838",
        size=14
    ),

    hovermode="x unified",

    # 3. Axis titles enlarged to size 16
    xaxis=dict(
        showgrid=False,
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16)) 
    ),
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))
    ),
    
    # Horizontal legend
    legend=dict(
        title=None, 
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="right",
        x=1,
        font=dict(color="#383838", size=14)
    )
)

fig_fossil.show()

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
fig_fossil.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\02_global_fossil_shares.html")



<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px; margin-bottom: 20px;">

<span style="font-size:22px; font-weight:700;">
Inside the Fossil Fuel Dominance<br>
</span>

<br>

<span style="font-size:18px;">
This stacked area chart breaks down the composition of the massive fossil fuel block.<br>
It highlights that within the ~80% of global energy still derived from fossil sources, Oil remains the dominant fuel, closely followed by Coal and Natural Gas.<br>
Despite a slight percentage decrease in the overall fossil share in recent years, the persistent and massive reliance on these three highly carbon-intensive fuels underscores the sheer scale of the global decarbonization challenge.
</span>

</div>

In [196]:
# Zoom on Low-carbon

df_lowcarbon = df_shares[[
    "nuclear_share_energy",
    "hydro_share_energy",
    "solar_share_energy",
    "wind_share_energy",
    "other_renewables_share_energy",
    "biofuel_share_energy"]]

fig_low = px.area(
    df_lowcarbon,
    x=df_lowcarbon.index,
    y=df_lowcarbon.columns,
    title="Global Low-Carbon Energy Shares",
    color_discrete_map={
        "nuclear_share_energy": "#9467bd",
        "hydro_share_energy": "#1f77b4",
        "solar_share_energy": "#f1c40f",
        "wind_share_energy": "#2ca02c",
        "other_renewables_share_energy": "#17becf",
        "biofuel_share_energy": "#8c564b"})

# --- CLEAN LEGEND NAMES ---
clean_names = {
    "nuclear_share_energy": "Nuclear",
    "hydro_share_energy": "Hydro",
    "solar_share_energy": "Solar",
    "wind_share_energy": "Wind",
    "other_renewables_share_energy": "Other renewables",
    "biofuel_share_energy": "Biofuel"}

fig_low.for_each_trace(lambda t: t.update(
    name=clean_names[t.name],
    legendgroup=clean_names[t.name],
    hovertemplate=t.hovertemplate.replace(t.name, clean_names[t.name])))

# --- UNIFIED PREMIUM LAYOUT ---
fig_low.update_layout(

    # Centered and enlarged title
    title=dict(
        text="Global Low-Carbon Energy Shares",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838")),

    xaxis_title="Year",
    yaxis_title="Share (%)",

    # Background
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=580,
    template="plotly_white",

    # Global text style
    font=dict(color="#383838", size=14),

    hovermode="x unified",

    # Axes
    xaxis=dict(
        showgrid=False,
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))),
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838"),
        title=dict(font=dict(color="#383838", size=16))),

    # Legend (NO TITLE)
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="right",
        x=1,
        font=dict(color="#383838", size=14)))

fig_low.show()

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
fig_low.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\03_global_ren_shares.html")



<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Shifting Makeup of Clean Energy<br>
</span>

<br>

<span style="font-size:18px;">
This chart details the internal evolution of low-carbon energy sources, which together still account for less than 20% of the global mix.<br>
Historically, Nuclear (purple) and Hydroelectric power (blue) provided the solid foundation of the world's clean energy base.<br>
However, a new dynamic is clearly visible in recent years: the relative share of nuclear power has gradually declined, while the recent overall expansion in the clean energy sector is almost entirely driven by the rapid, accelerating deployment of modern renewables like Wind (green) and Solar (yellow).
</span>

</div>


In [197]:
# 1. Define sources and categories based on the codebook
# All consumption metrics are in TWh [1-9]
sources_map = {
    "coal_consumption": ("Fossil Fuels", "Coal"),
    "oil_consumption": ("Fossil Fuels", "Oil"),
    "gas_consumption": ("Fossil Fuels", "Gas"),
    "solar_consumption": ("Renewables", "Solar"),
    "wind_consumption": ("Renewables", "Wind"),
    "hydro_consumption": ("Renewables", "Hydro"),
    "biofuel_consumption": ("Renewables", "Biofuels"),
    "other_renewable_consumption": ("Renewables", "Other Renewables"),
    "nuclear_consumption": ("Nuclear", "Nuclear")
}

# 2. Filter for the 2000-2024 period and real countries [10, 11]
df_period = owid_global[
    (owid_global["year"].between(2000, 2024)) & 
    (owid_global["iso_code"].notna())].copy()

# 3. Aggregate data for the TOTAL period (Sum of TWh, Mean of Efficiency)
treemap_data = []
for col, (cat, label) in sources_map.items():
    # Total energy consumed in the entire 24-year span [10]
    total_twh = df_period[col].sum()
    # Average economic efficiency for the period [12]
    avg_efficiency = df_period["energy_per_gdp"].mean()
    
    treemap_data.append({
        "Category": cat,
        "Source": label,
        "Total_TWh": total_twh,
        "Avg_kWh_per_GDP": avg_efficiency})

df_static = pd.DataFrame(treemap_data)

In [198]:
# 4. Create Treemap
fig = px.treemap(
    df_static,
    path=["Category", "Source"], 
    values="Total_TWh",
    color="Category",
    hover_data={
        "Total_TWh": ":.2f",
        "Avg_kWh_per_GDP": ":.2f"
    },
    title="Global Energy Consumption Mix",
    color_discrete_map={
        "Fossil Fuels": "orange",     
        "Renewables": "green",       
        "Nuclear": "royalblue"},
    labels={
        "Category": "",
        "Source": ""},
    template="plotly_white")

# --- UNIFIED PREMIUM LAYOUT ---
fig.update_layout(

    # Centered and enlarged title
    title=dict(
        text="Global Energy Consumption Mix",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838")),

    # Background
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    height=650,
    template="plotly_white",

    # Global text style
    font=dict(color="#383838", size=14),

    # Legend (NO TITLE)
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="right",
        x=1,
        font=dict(color="#383838", size=14)))

# Refine labels to show share and absolute volume
fig.update_traces(textinfo="label+percent entry+value")

fig.show()

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
fig.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\04_global_tree.html")


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px; margin-bottom: 20px;">

<span style="font-size:22px; font-weight:700;">
The Current Global Energy Landscape<br>
</span>

<br>

<span style="font-size:18px;">
This treemap provides a clear, proportional snapshot of the world's current energy consumption mix.<br>
The vast size of the orange "Fossil Fuels" rectangle visually emphasizes its continued stronghold on the global economy, with Oil and Gas alone accounting for over half of all energy used.<br>
In stark contrast, the green "Renewables" and blue "Nuclear" sections occupy a much smaller footprint. While modern sources like Solar and Wind are growing rapidly in media attention and capacity, this chart grounds us in the reality that they still represent a very small fraction of total global energy demand.
</span>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">


<div style="text-align:center;">

<h2 style="color:#003366;">"FOCUS ON WORLD BANK INCOME GROUPS</h2>

<p style="color:#777777;">
I transitioned from a broad global overview to a segmented analysis using official World Bank income groups to better reflect global economic diversity<br>
This methodology follows the Our World in Data standard for constructing regional aggregates and calculating energy efficiency metrics relative to GDP
</p>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

In [199]:
# 1. DATA PREPARATION
df_evo = owid_completed[owid_completed["iso_code"].notna()].copy()

# Fill missing values for energy consumption columns
cols_energy = ["fossil_fuel_consumption", "nuclear_consumption", "renewables_consumption"]
for col in cols_energy:
    df_evo[col] = df_evo[col].fillna(0)

# 2. AGGREGATION
df_stats = df_evo.groupby(["year", "income_group"]).agg({
    "fossil_fuel_consumption": "sum",
    "nuclear_consumption": "sum",
    "renewables_consumption": "sum",
    "iso_code": "nunique"
}).reset_index()

# 3. PERCENTAGE OF COUNTRIES
# Calculate the share of countries belonging to each income group per year
total_states_per_year = df_stats.groupby("year")["iso_code"].transform("sum")
df_stats["state_share_pct"] = (df_stats["iso_code"] / total_states_per_year * 100).round(0)

# 4. FIXED ORDER 
# Consistent with other charts: from High to Low income
income_order = ["High-income countries", "Upper-middle-income countries", "Lower-middle-income countries", "Low-income countries"]

# 5. BASE STACKED BAR CHART
fig_br = px.bar(
    df_stats,
    x="income_group",
    y=cols_energy,
    animation_frame="year",
    title="Global Energy Transition: High to Low Income Perspective",
    labels={
        "value": "Primary Energy Consumption (TWh)",
        "income_group": "Income Group",
        "variable": "Energy Source"
    },
    category_orders={
        "income_group": income_order,
        "variable": cols_energy # Stack order: Fossils at the bottom, Renewables at the top
    },
    template="plotly_white",
    color_discrete_map={
        "fossil_fuel_consumption": "orange", # Premium Orange
        "nuclear_consumption": "royalblue",     # Blue
        "renewables_consumption": "green"   # Green
    }
)

# --- CLEAN LEGEND NAMES ---
clean_names = {
    "fossil_fuel_consumption": "Fossil fuels",
    "nuclear_consumption": "Nuclear",
    "renewables_consumption": "Renewables"
}
fig_br.for_each_trace(lambda t: t.update(name=clean_names.get(t.name, t.name)))

# 6. ADD PERCENTAGE LABELS TO ALL FRAMES (Using color #383838)
for frame in fig_br.frames:
    year = int(frame.name)
    df_year = df_stats[df_stats["year"] == year]
    
    # Add text layer for percentages to each frame
    frame.data += (
        go.Scatter(
            x=df_year["income_group"],
            y=df_year[cols_energy].sum(axis=1),
            mode="text",
            text=df_year["state_share_pct"].astype(str) + "%",
            textposition="top center",
            textfont=dict(size=16, color="#383838"), # <--- REQUIRED TEXT COLOR
            showlegend=False
        ),
    )

# Add labels to the initial frame
df_initial = df_stats[df_stats["year"] == df_stats["year"].min()]
fig_br.add_trace(
    go.Scatter(
        x=df_initial["income_group"],
        y=df_initial[cols_energy].sum(axis=1),
        mode="text",
        text=df_initial["state_share_pct"].astype(str) + "%",
        textposition="top center",
        textfont=dict(size=16, color="#383838"), # <--- REQUIRED TEXT COLOR
        showlegend=False
    )
)

# 7. PREMIUM LAYOUT (Unified text colors to #383838)
# Automatic Y-axis range + 15% margin for labels
max_val = df_stats[cols_energy].sum(axis=1).max() * 1.15

fig_br.update_layout(
    barmode='stack',
    height=800,
    font=dict(family="Arial", color="#383838", size=14), # <--- GLOBAL FONT COLOR
    
    title=dict(
        text="Global Energy Transition: High to Low Income Perspective<br><sup>Percentages indicate the share of countries in each group</sup>",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838") # <--- TITLE COLOR
    ),

    # Quarto-compatible background colors
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",

    xaxis=dict(
        title=dict(text="World Bank Income Groups", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"), # <--- X TICK COLOR
        showgrid=False
    ),
    yaxis=dict(
        title=dict(text="Primary Energy Consumption (TWh)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"), # <--- Y TICK COLOR
        gridcolor="rgba(56, 56, 56, 0.1)", # Very subtle gray for the grid
        range=[0, max_val]
    ),

    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="right",
        x=1,
        font=dict(size=14, color="#383838") # <--- LEGEND COLOR
    )
)

# Save
fig_br.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\05_income_bar.html")

# Show
fig_br.show()


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
Energy Consumption Across Income Groups: The Development Challenge<br>
</span>

<br>

<span style="font-size:18px;">
This animated bar chart reveals the stark reality of global economic development and its historical reliance on fossil fuels over the last two decades.<br>
As the timeline progresses, we see a dramatic surge in total energy consumption, particularly within the "Upper-middle-income" group, as developing nations rapidly industrialize to raise their citizens' living standards.<br>
Crucially, the massive, expanding orange bars demonstrate that this explosive new energy demand has been met almost entirely by fossil fuels. While "High-income" nations show a larger integration of nuclear and renewables (blue and green), the animation clearly underscores that the current, proven pathway to economic prosperity remains deeply carbon-intensive.
</span>

</div>


In [200]:
# 1. DATA PREPARATION
df_income_map = owid_completed[
    (owid_completed["iso_code"].notna()) &
    (owid_completed["income_group"].notna()) &
    (owid_completed["year"].between(2000, 2024))
].copy()

# Clean invisible trailing spaces
df_income_map["income_group"] = df_income_map["income_group"].astype(str).str.strip()

# Normalize OWID names to match your palette
df_income_map["income_group"] = df_income_map["income_group"].replace({
    "High-income countries": "High income",
    "Upper-middle-income countries": "Upper-middle income",
    "Lower-middle-income countries": "Lower-middle income",
    "Low-income countries": "Low income"
})

# Ensure chronological sorting for smooth animation
df_income_map["year"] = df_income_map["year"].astype(int)
df_income_map = df_income_map.sort_values(by=["year", "iso_code"])

# 2. CATEGORY ORDER
income_order = [
    "High income",
    "Upper-middle income",
    "Lower-middle income",
    "Low income",
    "Not classified"
]

# 3. ANIMATED CHOROPLETH MAP
color_map = {
    "High income": "#003f5c",
    "Upper-middle income": "#457591",
    "Lower-middle income": "#92a673",
    "Low income": "#d4b83b",
    "Not classified": "#8a8a8a"
}

fig = px.choropleth(
    df_income_map,
    locations="iso_code",
    color="income_group",
    hover_name="country",
    animation_frame="year",
    category_orders={"income_group": income_order},
    color_discrete_map=color_map,
    labels={"income_group": ""},
    template="plotly_white"
)

# 4. PREMIUM LAYOUT + GEOGRAPHIC STYLE
fig.update_layout(

    # Title (centered)
    title=dict(
        text="Global Evolution: World Bank Income Groups",
        x=0.5,
        xanchor='center',
        font=dict(size=24, color="#383838")
    ),

    # Global text style
    font=dict(
        color="#383838",
        size=14
    ),

    # Background (premium cream)
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",

    # Legend (horizontal, centered)
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(color="#383838", size=14)
    ),

    height=750,
    margin={"r": 0, "t": 80, "l": 0, "b": 0},

    # Geographic styling
    geo=dict(
        bgcolor="#FFF8E7",
        showframe=False,
        showcoastlines=True,
        coastlinecolor="#383838",
        showocean=True,
        oceancolor="#f4e8e8",
        projection_type="natural earth"
    )
)

fig.show()

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
fig.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\06_income_map.html")


<div style="text-align:center;">
<h3 style="color:GOLD">Global Energy Transition: Economic Efficiency vs. Carbon Intensity (2000–2024)</h3>

In [201]:
# ---------------------------------------------------------
# 1. HISTORICAL AGGREGATION (2000–2024)
# ---------------------------------------------------------
# Compute long‑term averages per country across the full period
df_historical = (
    owid_completed[owid_completed["year"].between(2000, 2024)]
    .groupby("country")[["energy_per_gdp", "carbon_intensity_elec"]]
    .mean()
    .reset_index()
    .rename(columns={
        "energy_per_gdp": "avg_energy_per_gdp",
        "carbon_intensity_elec": "avg_carbon_intensity"}))

# ---------------------------------------------------------
# 2. BASE DATA (2000–2024)
# ---------------------------------------------------------
# Filter base dataset and keep only valid rows
df_all = owid_completed[
    (owid_completed["year"].between(2000, 2024)) &
    (owid_completed["iso_code"].notna())][[
    "country",
    "year",
    "energy_per_gdp",
    "carbon_intensity_elec",
    "population",
    "income_group"]]

df_all = (
    df_all
    .dropna(subset=["energy_per_gdp", "carbon_intensity_elec", "population"])
    .query("population > 0"))

# Merge with historical averages
df_quadrant = df_all.merge(df_historical, on="country", how="inner")

# ---------------------------------------------------------
# 3. GLOBAL HISTORICAL MEDIANS (2000–2024)
# ---------------------------------------------------------
x_median = df_all["energy_per_gdp"].median()
y_median = df_all["carbon_intensity_elec"].median()


In [202]:
# 1. DATA PREPARATION
df_quadrant = df_quadrant.copy()
df_quadrant["income_group"] = df_quadrant["income_group"].astype(str).str.strip().replace({
    "High-income countries": "High income",
    "Upper-middle-income countries": "Upper-middle income",
    "Lower-middle-income countries": "Lower-middle income",
    "Low-income countries": "Low income"
})

# Define shared income colors
income_colors = {
    "High income": "#003f5c",
    "Upper-middle income": "#457591",
    "Lower-middle income": "#92a673",
    "Low income": "#d4b83b",
    "Not classified": "#8a8a8a"
}

# 2. CALCULATE MEDIANS & FIXED RANGES
x_median = df_quadrant["energy_per_gdp"].median()
y_median = df_quadrant["carbon_intensity_elec"].median()
max_x = df_quadrant["energy_per_gdp"].max() * 1.1
max_y = df_quadrant["carbon_intensity_elec"].max() * 1.1

# 3. ANIMATED SCATTER PLOT
fig = px.scatter(
    df_quadrant,
    x="energy_per_gdp",
    y="carbon_intensity_elec",
    size="population",
    size_max=80,
    animation_frame="year",
    animation_group="country",
    hover_name="country",
    color="income_group",
    color_discrete_map=income_colors,
    range_x=[0, max_x],
    range_y=[0, max_y],
    labels={
        "energy_per_gdp": "Energy per GDP (kWh per $)",
        "carbon_intensity_elec": "Carbon Intensity (gCO₂/kWh)",
        "income_group": "Income Group"
    },
    template="plotly_white"
)

# 4. QUADRANT LINES (Grey for visibility on cream)
fig.add_vline(x=x_median, line_width=1.5, line_dash="dash", line_color="#383838", opacity=0.4)
fig.add_hline(y=y_median, line_width=1.5, line_dash="dash", line_color="#383838", opacity=0.4)

# 5. PREMIUM LAYOUT
fig.update_layout(
    title=dict(
        text="Energy Efficiency vs Carbon Intensity",
        x=0.5, xanchor="center",
        font=dict(size=24, color="#383838")
    ),
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    font=dict(size=14, color="#383838"),
    xaxis=dict(gridcolor="rgba(0,0,0,0.08)"),
    yaxis=dict(gridcolor="rgba(0,0,0,0.08)"),
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
    height=800,
    margin=dict(t=120)
)

fig.update_traces(marker=dict(opacity=0.7, line=dict(width=0.5, color="white")))

# 6. SAVE AND SHOW
output_path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\07_income_scatter.html"
fig.write_html(output_path)
fig.show()


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Dual Challenge: Energy Efficiency vs. Carbon Intensity<br>
</span>

<br>

<span style="font-size:18px;">
This animated scatter plot dissects the core mechanics of the global energy transition. It tracks countries across two critical metrics: <b>Energy per GDP</b> on the X-axis (economic efficiency) and <b>Carbon Intensity</b> on the Y-axis (how polluting the energy mix is). The ultimate goal for every nation is the bottom-left quadrant—highly efficient economies powered by clean energy.<br><br>
Watching the animation over two decades reveals a clear and powerful trend: a massive, collective drift of the bubbles to the <i>left</i>. This represents a global triumph in energy efficiency; across all income groups, nations are successfully generating more economic wealth (GDP) for every unit of energy they consume.<br><br>
However, the lack of a strong, unified <i>downward</i> movement exposes the difficult reality of the climate crisis. While we have learned to use energy much more efficiently to build our economies, the actual energy we are burning remains stubbornly carbon-intensive, particularly in rapidly industrializing middle-income nations. We are doing more with less, but the "less" is still mostly fossil fuels.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">


<div style="text-align:center;">

<h2 style="color:#003366;">FOCUS ON SINGULAR COUNTRY: EXPLORING UNIQUE ENERGY PATHWAYS</h2>

<p style="color:#777777;">
I am now narrowing the focus to individual country behaviors, where you will find a variety of insights and curiosities regarding how different nations approach their energy transitions.<br>
By examining several Top 25 rankings across renewable, fossil, and nuclear sectors, this section uncovers the specific strategies that allow certain countries<br>
to emerge as "Sustainable Leaders" while others remain fossil-dependent.
</p>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">



<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Evolution of Energy Consumption by Source: Top 25 Countries (2000-2024)
</h2>

In [203]:
# ---------------------------------------------------------
# 1. DATA PREPARATION & FLAG MAP
# ---------------------------------------------------------
# Map ISO3 country codes to emoji flags (add more if needed)
iso3_to_flag = {
    'CHN': '🇨🇳', 'USA': '🇺🇸', 'IND': '🇮🇳', 'RUS': '🇷🇺', 'JPN': '🇯🇵',
    'CAN': '🇨🇦', 'DEU': '🇩🇪', 'KOR': '🇰🇷', 'BRA': '🇧🇷', 'IRN': '🇮🇷',
    'SAU': '🇸🇦', 'FRA': '🇫🇷', 'IDN': '🇮🇩', 'GBR': '🇬🇧', 'MEX': '🇲🇽',
    'ITA': '🇮🇹', 'TUR': '🇹🇷', 'AUS': '🇦🇺', 'ESP': '🇪🇸', 'POL': '🇵🇱',
    'ZAF': '🇿🇦', 'THA': '🇹🇭', 'VNM': '🇻🇳', 'EGY': '🇪🇬', 'ARG': '🇦🇷',
    'MYS': '🇲🇾', 'PAK': '🇵🇰', 'ARE': '🇦🇪', 'NLD': '🇳🇱', 'DZA': '🇩🇿',
    'PHL': '🇵🇭', 'KAZ': '🇰🇿', 'BEL': '🇧🇪', 'SWE': '🇸🇪', 'VEN': '🇻🇪',
    'COL': '🇨🇴', 'CHL': '🇨🇱', 'IRQ': '🇮🇶', 'BGD': '🇧🇩', 'CZE': '🇨🇿',
    'ROU': '🇷🇴', 'PER': '🇵🇪', 'GRC': '🇬🇷', 'PRT': '🇵🇹', 'NGA': '🇳🇬',
    'HUN': '🇭🇺', 'AUT': '🇦🇹', 'CHE': '🇨🇭', 'UKR': '🇺🇦', 'NOR': '🇳🇴'
}

# Keep only valid countries (ISO3 codes)
df_countries = owid_completed[owid_completed["iso_code"].str.len() == 3].copy()

# Fill missing values
df_countries["fossil_fuel_consumption"] = df_countries["fossil_fuel_consumption"].fillna(0)
df_countries["renewables_consumption"] = df_countries["renewables_consumption"].fillna(0)
df_countries["nuclear_consumption"] = df_countries["nuclear_consumption"].fillna(0)

# Compute total primary energy
df_countries["total_energy"] = (
    df_countries["fossil_fuel_consumption"] +
    df_countries["renewables_consumption"] +
    df_countries["nuclear_consumption"]
)

# Create the Y-axis label: Flag + Country Name
df_countries["country_label"] = (
    df_countries["iso_code"].map(iso3_to_flag).fillna("🏳️") 
    + " " + df_countries["country"]
)

# Filter years 2000–2024
df_countries = df_countries[
    (df_countries["year"] >= 2000) & (df_countries["year"] <= 2024)
]

# Top 25 countries based on the latest year
df_latest = df_countries[df_countries["year"] == 2024].copy()
df_top25 = df_latest.nlargest(25, "total_energy").copy()
df_top25 = df_top25.sort_values("total_energy", ascending=True)

# Order based on the new label (Flag + Country)
country_order = df_top25["country_label"].tolist()

# Prepare the dataset for plotting
df_plot = df_countries[df_countries["country_label"].isin(country_order)].copy()
df_plot = df_plot.sort_values(["year", "country_label"])

# ---------------------------------------------------------
# 2. TRANSFORM TO LONG FORMAT FOR PLOTLY
# ---------------------------------------------------------
# NOTE: Keep iso_code and total_energy for tooltip display
df_long = df_plot.melt(
    id_vars=["country_label", "iso_code", "year", "total_energy"],
    value_vars=["fossil_fuel_consumption", "renewables_consumption", "nuclear_consumption"],
    var_name="source",
    value_name="consumption"
)

# ---------------------------------------------------------
# 3. COLOR PALETTE
# ---------------------------------------------------------
colors = {
    "fossil_fuel_consumption": "orange",
    "renewables_consumption": "green",
    "nuclear_consumption": "royalblue"
}

# ---------------------------------------------------------
# 4. ANIMATED STACKED BAR CHART
# ---------------------------------------------------------
max_x_value = df_plot["total_energy"].max() * 1.05

fig = px.bar(
    df_long,
    x="consumption",
    y="country_label",
    color="source",
    orientation="h",
    animation_frame="year",
    animation_group="country_label",
    color_discrete_map=colors,
    custom_data=["iso_code", "total_energy"],  # Extra tooltip data
    title="Energy Mix Evolution 2000–2024 (Top 25)",
    category_orders={"country_label": country_order},
    range_x=[0, max_x_value]
)

# ---------------------------------------------------------
# 5. CLEAN LEGENDS & CUSTOM TOOLTIP
# ---------------------------------------------------------
clean_names = {
    "fossil_fuel_consumption": "Fossil fuels",
    "renewables_consumption": "Renewables",
    "nuclear_consumption": "Nuclear"
}

fig.for_each_trace(lambda t: t.update(
    name=clean_names.get(t.name, t.name),
    legendgroup=clean_names.get(t.name, t.name),
    hovertemplate=(
        "<b>ISO Code:</b> %{customdata[0]}<br>"
        "<b>This Category:</b> %{x:.0f} TWh<br>"
        "<b>Total Country Energy:</b> %{customdata[1]:.0f} TWh"
        "<extra>%{data.name}</extra>"
    )
))

# ---------------------------------------------------------
# 6. PREMIUM LAYOUT
# ---------------------------------------------------------
fig.update_layout(
    barmode="stack",
    height=750,
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    title=dict(text="Energy Mix Evolution 2000–2024 (Top 25)", x=0.5, font=dict(size=24)),
    legend=dict(title=None, orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1),
    xaxis=dict(title="Energy Consumption (TWh)", gridcolor="rgba(0,0,0,0.08)"),
    yaxis=dict(title="", autorange="reversed"),
    margin=dict(l=10, r=40, t=80, b=40)
)

# Smooth animation
if hasattr(fig.layout, "updatemenus") and len(fig.layout.updatemenus) > 0:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 600
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

fig.show()

# --- SAVE THE FIGURE INTO A SPECIFIC FOLDER ---
fig.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\08_countries_barh.html")

<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Heavyweights: Top 25 Energy Consumers<br>
</span>

<br>

<span style="font-size:18px;">
This animated bar chart ranks the world's top 25 energy consumers, providing a stark visualization of the massive scale and shifting center of gravity in global energy demand.<br><br>
The defining feature of the last two decades is the unparalleled, explosive growth of China. While advanced economies like the United States show stabilizing total energy use—and several European nations even show slight declines while slowly substituting fossil fuels (orange) for renewables (green)—the rapid industrialization of China and India has driven massive absolute increases in global fossil fuel consumption.<br><br>
However, the final snapshot in 2024 also highlights an important duality: while China burns vastly more fossil fuels than any other nation, the sheer scale of its recent investments in clean energy means that its green "renewables" block alone is now larger than the total energy consumption of many fully developed countries.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
"Wealth vs Fossil Dependency (Top 50 Emitters)"
<h2

In [204]:
# ---------------------------------------------------------
# 1. DATA PREPARATION
# ---------------------------------------------------------
df_bubble = owid_completed[owid_completed["iso_code"].notna()].copy()

# Calculate new metrics (GDP per Capita and Emissions per Capita)
df_bubble["gdp_per_capita"] = df_bubble["gdp"] / df_bubble["population"]

# Multiply by 1,000,000 to convert from Millions of Tonnes to actual Tonnes per capita (t)
df_bubble["emissions_per_capita"] = (df_bubble["greenhouse_gas_emissions"] * 1000000) / df_bubble["population"]

# Drop zero or negative values that break logarithmic charts
df_bubble = df_bubble[(df_bubble["gdp_per_capita"] > 0) & (df_bubble["fossil_energy_per_capita"] > 0)]

# Select Top 50 Absolute Emitters to have a comprehensive but clean view
top_countries = df_bubble.groupby("country")["greenhouse_gas_emissions"].sum().nlargest(50).index
df_bubble = df_bubble[df_bubble["country"].isin(top_countries)]

# Chronological sorting is CRITICAL for smooth animations
df_bubble["year"] = df_bubble["year"].astype(int)
df_bubble = df_bubble.sort_values(by=["year", "country"]).dropna(
    subset=["gdp_per_capita", "fossil_energy_per_capita", "greenhouse_gas_emissions", "emissions_per_capita"]
)

# ---------------------------------------------------------
# 2. LOCKING THE AXES (Calculate global limits)
# ---------------------------------------------------------
# We use a small margin (* 0.8 for min, * 1.2 for max) so bubbles don't get cut off
min_gdp = df_bubble["gdp_per_capita"].min() * 0.8
max_gdp = df_bubble["gdp_per_capita"].max() * 1.2

min_fossil = df_bubble["fossil_energy_per_capita"].min() * 0.8
max_fossil = df_bubble["fossil_energy_per_capita"].max() * 1.2

# ---------------------------------------------------------
# 3. ANIMATED BUBBLE CHART
# ---------------------------------------------------------
fig = px.scatter(
    df_bubble,
    x="gdp_per_capita",            
    y="fossil_energy_per_capita",   
    animation_frame="year",
    animation_group="country",
    size="greenhouse_gas_emissions", 
    color="emissions_per_capita",    
    hover_name="country",
    size_max=60,
    log_x=True,                     
    log_y=True,                       
    trendline="ols",                          
    trendline_options=dict(log_x=True, log_y=True),   
    range_x=[min_gdp, max_gdp],      
    range_y=[min_fossil, max_fossil], 
    color_continuous_scale=["#FFEA00", "#FF0000", "#8B0000", "#000000"], 
    labels={
        "gdp_per_capita": "GDP per Capita ($)",
        "fossil_energy_per_capita": "Fossil Energy per Capita (kWh)",
        "greenhouse_gas_emissions": "Total GHG Emissions",
        "emissions_per_capita": "GHG per Capita (Tonnes)"
    }
)

# ---------------------------------------------------------
# 4. PREMIUM LAYOUT
# ---------------------------------------------------------
fig.update_layout(
    height=800,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    
    # Title with Subtitle using HTML tags
    title=dict(
        text=(
            "Wealth vs Fossil Dependency (Top 50 Emitters)<br>"
            "<span style='font-size:16px; color:#666666; font-weight:normal;'>"
            "Bubble size represents Total Absolute GHG Emissions"
            "</span>"
        ),
        x=0.5, 
        xanchor="center", 
        font=dict(size=24, color="#383838")
    ),

    xaxis=dict(
        title=dict(font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title=dict(font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),

    coloraxis_colorbar=dict(
        title=dict(text="GHG per Capita (t)", font=dict(color="#383838", size=14)),
        tickfont=dict(color="#383838")
    ),
    margin=dict(l=80, r=40, t=80, b=40)
)

# Style the bubbles (white borders)
fig.update_traces(
    marker=dict(line=dict(width=0.5, color="white"), opacity=0.7),
    selector=dict(mode="markers")
)

# Style the trendline (make it red and dashed for a premium look)
fig.update_traces(
    line=dict(color="red", width=2, dash="dash"), 
    selector=dict(mode="lines")
)

# Animation Smoothness
if hasattr(fig.layout, "updatemenus") and len(fig.layout.updatemenus) > 0:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

fig.show()

# Save 
fig.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\09_countries_scatter_fossil.html")


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
Wealth vs. Fossil Dependency: The Decoupling Challenge<br>
</span>

<br>

<span style="font-size:18px;">
This animated chart illustrates the historical positive correlation between economic wealth (GDP per Capita) and fossil fuel dependency.<br><br>
As nations grow richer and move to the right, their per‑capita fossil consumption and resulting emissions (represented by the larger bubbles and darker, more intense colors) tend to rise sharply along the red trendline.<br><br>
The ultimate challenge of the global energy transition is <i>decoupling</i>—breaking this historical trendline so that countries can continue to grow and prosper economically while simultaneously reducing their reliance on fossil fuels.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Renewable Transition and Economic Performance (2000–2024)</b><br>Evolution of new Renewables and Carbon Intensity
<h2


In [205]:
# ---------------------------------------------------------
# 1. DATA PREPARATION AND CLEANING (2000-2024)
# ---------------------------------------------------------
df_plot = owid_completed[
    (owid_completed["iso_code"].notna()) & 
    (owid_completed["year"].between(2000, 2024))
].sort_values(["country", "year"]).copy()

# ---------------------------------------------------------
# 2. CUMULATIVE METRICS
# ---------------------------------------------------------
df_plot["yearly_new_tech_pc"] = (
    df_plot["solar_elec_per_capita"].fillna(0) + 
    df_plot["wind_elec_per_capita"].fillna(0) + 
    df_plot["other_renewables_elec_per_capita"].fillna(0)
)

# Calculate cumulative sums for each country
df_plot["cumulative_new_tech_pc"] = df_plot.groupby("country")["yearly_new_tech_pc"].cumsum()
df_plot["cumulative_renovable_pc"] = df_plot.groupby("country")["renewables_elec_per_capita"].cumsum()

# Carbon Intensity (Color - Expanding Mean to smooth out spikes)
df_plot["avg_carbon_intensity"] = (
    df_plot.groupby("country")["carbon_intensity_elec"]
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)

# Economic Gain: proxy for $/kWh efficiency (Prevent division by zero)
df_plot = df_plot[df_plot["energy_per_gdp"] > 0]
df_plot["economic_gain"] = 1 / df_plot["energy_per_gdp"]

# ---------------------------------------------------------
# 3. HANDLE MISSING VALUES & TOP 50 FILTER
# ---------------------------------------------------------
df_plot = df_plot.dropna(subset=[
    "renewables_share_energy", 
    "economic_gain", 
    "avg_carbon_intensity",
    "cumulative_renovable_pc"
])

# Remove values <= 0 for Logarithmic Scales
df_plot = df_plot[(df_plot["cumulative_renovable_pc"] > 0) & (df_plot["economic_gain"] > 0)]

# Select Top 50 countries by maximum cumulative renewables
top_countries = df_plot.groupby("country")["cumulative_renovable_pc"].max().nlargest(50).index
df_plot = df_plot[df_plot["country"].isin(top_countries)]

# ---------------------------------------------------------
# 4. FIX AXES AND COLORS
# ---------------------------------------------------------
color_min = df_plot["avg_carbon_intensity"].quantile(0.05)
color_max = df_plot["avg_carbon_intensity"].quantile(0.95)

# Lock the axes using Logarithmic margins to prevent jumping
x_range = [df_plot["cumulative_renovable_pc"].min() * 0.8, df_plot["cumulative_renovable_pc"].max() * 1.2]
y_range = [df_plot["economic_gain"].min() * 0.8, df_plot["economic_gain"].max() * 1.2]

# ---------------------------------------------------------
# 5. CREATE ANIMATED SCATTER PLOT
# ---------------------------------------------------------
fig = px.scatter(
    df_plot,
    x="cumulative_renovable_pc",
    y="economic_gain",
    animation_frame="year",
    animation_group="country",
    size="cumulative_new_tech_pc",
    color="avg_carbon_intensity",
    range_color=[color_min, color_max],
    color_continuous_scale="RdYlGn_r", # Brilliant "Traffic Light" Color scale: Red -> Green
    hover_name="country",
    hover_data={"country": False, "renewables_share_energy": True},
    trendline="ols",
    trendline_options=dict(log_x=True, log_y=True),
    size_max=80,
    log_x=True,                        # <--- LOG SCALE added for consistency and outlier management
    log_y=True,                        # <--- LOG SCALE added 
    range_x=x_range,                   
    range_y=y_range                    
)

# ---------------------------------------------------------
# 6. PREMIUM LAYOUT
# ---------------------------------------------------------
fig.update_layout(
    height=850,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),

    # HTML Subtitle consistent with previous charts
    title=dict(
        text=(
            "Renewable Transition and Economic Performance<br>"
            "<span style='font-size:16px; color:#666666; font-weight:normal;'>"
            "Size: Technological Effort | Color: Carbon Intensity (Red to Green)"
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=24, color="#383838")
    ),
    
    xaxis=dict(
        title=dict(text="Cumulative Renewable Production per Capita (kWh)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title=dict(text="Economic Gain ($ per kWh of Energy)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),

    coloraxis_colorbar=dict(
        title=dict(text="Average<br>CO₂ Intensity<br>(gCO₂e/kWh)", font=dict(color="#383838", size=14)),
        tickfont=dict(color="#383838")
    ),
    margin=dict(l=80, r=40, t=100, b=50)
)

# White borders for elegance and MINIMUM SIZE
fig.update_traces(
    marker=dict(
        line=dict(width=1, color="grey"), 
        opacity=0.85,
        sizemin=6   
    ),
    selector=dict(mode="markers")
)


# Animation Smoothness
if hasattr(fig.layout, "updatemenus") and len(fig.layout.updatemenus) > 0:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

# Show
fig.show()
# Save
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\10_final_economic_transition.html")

<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Renewable Frontier: Effort vs. Economic Return<br>
</span>

<br>

<span style="font-size:18px;">
This animated scatter plot explores the relationship between a country's historical commitment to green energy and its overall economic efficiency. The X-axis tracks "Cumulative Renewable Production per Capita," effectively measuring a nation's long-term experience and sustained investment in renewable technologies.<br><br>
As the animation progresses over the decades, we observe the bubbles moving to the right and growing larger (indicating increasing technological effort). Crucially, as they move right, the color of many bubbles shifts from warm colors to green, demonstrating that accumulating renewable capacity successfully and tangibly drives down overall carbon intensity.<br><br>
While the slightly downward-sloping trendline indicates that the absolute highest per-capita producers of renewables don't necessarily extract the highest GDP per unit of energy (often due to specific, highly energy-intensive industries located in those progressive nations), the chart clearly illustrates a growing global momentum toward a cleaner, technology-driven energy frontier.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
    Energy Analysis by Country and Per Capita
</h2>

<h3 style="text-align:center; color:grey; font-size:18px; line-height:1.5;">
    This table lists all OWID indicators that are measured on a per‑capita basis,<br>
    meaning they express energy consumption, production, or electricity generation divided by the population.<br>
    These metrics allow fair comparisons between countries by showing how much energy each person uses on average,regardless of national size.<br>
    Every value is expressed in :
</h3>

<h2 style="text-align:center; color:green; font-size:22px; margin-top:5px;">
    kilowatt‑hours per person
</h2>

<h3 style="text-align:center; color:grey; font-size:18px; margin-top:0;">
    making the indicators directly comparable across sources and technologies.
</h3>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Renewable Electricity Growth Analysis (2000–2024): Country-Level Comparison
</h2>


In [206]:
# ---------------------------------------------------------
# 1. Filter valid countries (non-null ISO codes)
# ---------------------------------------------------------
df_countries = owid_completed[owid_completed["iso_code"].notna()]

# ---------------------------------------------------------
# 2. Select the period 2000–2024
# ---------------------------------------------------------
df_period = df_countries[df_countries["year"].between(2000, 2024)]

# ---------------------------------------------------------
# 3. Define renewable electricity breakdown columns
# ---------------------------------------------------------
breakdown_cols = [
    "solar_elec_per_capita",
    "wind_elec_per_capita",
    "hydro_elec_per_capita",
    "biofuel_elec_per_capita",
    "other_renewables_elec_per_capita_exc_biofuel"]

# ---------------------------------------------------------
# 4. Extract snapshots for 2000 and 2024
# ---------------------------------------------------------
# AGGIUNTO "iso_code" QUI ↓
base_cols = ["country", "iso_code", "renewables_elec_per_capita"] + breakdown_cols

df_2000 = (
    df_period[df_period["year"] == 2000][base_cols]
    .rename(columns={"renewables_elec_per_capita": "ren_2000"}))

df_2024 = (
    df_period[df_period["year"] == 2024][base_cols]
    .rename(columns={"renewables_elec_per_capita": "ren_2024"}))

# ---------------------------------------------------------
# 5. Merge both years
# ---------------------------------------------------------
# AGGIUNTO "iso_code" COME CHIAVE DI MERGE ↓ per non farlo sdoppiare in _x e _y
df_growth = pd.merge(df_2000, df_2024, on=["country", "iso_code"], how="inner")

# ---------------------------------------------------------
# 6. Rename breakdown columns for clarity (_x = 2000, _y = 2024)
# ---------------------------------------------------------
for col in breakdown_cols:
    df_growth.rename(
        columns={
            f"{col}_x": f"{col}_2000",
            f"{col}_y": f"{col}_2024"
        },
        inplace=True)

# ---------------------------------------------------------
# 7. Compute total renewable growth
# ---------------------------------------------------------
df_growth["growth_2000_2024"] = df_growth["ren_2024"] - df_growth["ren_2000"]

# ---------------------------------------------------------
# 8. Compute growth for each renewable source
# ---------------------------------------------------------
for col in breakdown_cols:
    df_growth[f"{col}_growth"] = (
        df_growth[f"{col}_2024"] - df_growth[f"{col}_2000"])


# ---------------------------------------------------------
# 9. Select Top 25 countries by total renewable growth
# ---------------------------------------------------------
top25_growth = (
    df_growth.sort_values("growth_2000_2024", ascending=False).round(2)
    .head(25))

top25_growth.head()


,country,iso_code,ren_2000,solar_elec_per_capita_2000,wind_elec_per_capita_2000,hydro_elec_per_capita_2000,biofuel_elec_per_capita_2000,other_renewables_elec_per_capita_exc_biofuel_2000,ren_2024,solar_elec_per_capita_2024,wind_elec_per_capita_2024,hydro_elec_per_capita_2024,biofuel_elec_per_capita_2024,other_renewables_elec_per_capita_exc_biofuel_2024,growth_2000_2024,solar_elec_per_capita_growth,wind_elec_per_capita_growth,hydro_elec_per_capita_growth,biofuel_elec_per_capita_growth,other_renewables_elec_per_capita_exc_biofuel_growth
91,Iceland,ISL,27053.64,0.0,0.00,22361.03,0.00,4692.62,51436.60,0.00,25.37,36114.40,0.00,14999.28,24382.96,0.00,25.37,13753.37,0.00,10306.66
22,Bhutan,BTN,2998.97,0.0,0.00,2998.97,0.00,0.00,15039.07,0.00,0.00,15039.07,0.00,0.00,12040.09,0.00,0.00,12040.09,0.00,0.00
107,Laos,LAO,603.92,0.0,0.00,603.92,0.00,0.00,6158.48,10.29,0.00,6141.13,4.58,0.00,5554.56,10.29,0.00,5537.21,4.58,0.00
79,Greenland,GRL,3023.40,0.0,0.00,3023.40,0.00,0.00,8571.98,0.00,0.00,8571.98,0.00,0.00,5548.58,0.00,0.00,5548.58,0.00,0.00
53,Denmark,DNK,1043.14,0.0,794.06,5.62,243.46,0.00,5251.43,632.38,3434.59,3.35,1181.11,0.00,4208.29,632.38,2640.54,-2.27,937.65,0.00


In [207]:
# ---------------------------------------------------------
# 0. UPDATED MANUAL ISO TO FLAG MAPPING (Fossil + Renewables)
# ---------------------------------------------------------
iso3_to_flag = {
    # Countries from previous fossil chart
    "QAT": "🇶🇦", "SAU": "🇸🇦", "TWN": "🇹🇼", "BRN": "🇧🇳", "OMN": "🇴🇲",
    "CHN": "🇨🇳", "KOR": "🇰🇷", "TKM": "🇹🇲", "GIB": "🇬🇮", "SPM": "🇵🇲",
    "IRQ": "🇮🇶", "IRN": "🇮🇷", "TTO": "🇹🇹", "BHR": "🇧🇭", "NCL": "🇳🇨",
    "KAZ": "🇰🇿", "SGP": "🇸🇬", "MYS": "🇲🇾", "BIH": "🇧🇦", "SYC": "🇸🇨",
    "FRO": "🇫🇴", "LBY": "🇱🇾", "VNM": "🇻🇳", "ATG": "🇦🇬", "LAO": "🇱🇦",
    # New countries for renewables chart
    "ISL": "🇮🇸", "BTN": "🇧🇹", "GRL": "🇬🇱", "DNK": "🇩🇰", "FIN": "🇫🇮",
    "NLD": "🇳🇱", "DEU": "🇩🇪", "AUS": "🇦🇺", "URY": "🇺🇾", "PRT": "🇵🇹",
    "ESP": "🇪🇸", "EST": "🇪🇪", "GRC": "🇬🇷", "IRL": "🇮🇪", "BEL": "🇧🇪",
    "ALB": "🇦🇱", "LTU": "🇱🇹", "AUT": "🇦🇹", "GBR": "🇬🇧", "SWE": "🇸🇪",
    "CHL": "🇨🇱", "USA": "🇺🇸"
}

# ---------------------------------------------------------
# 10. Prepare data for the horizontal stacked bar chart
# ---------------------------------------------------------
growth_cols = [
    "solar_elec_per_capita_growth",
    "wind_elec_per_capita_growth",
    "hydro_elec_per_capita_growth",
    "biofuel_elec_per_capita_growth",
    "other_renewables_elec_per_capita_exc_biofuel_growth"
]

df_plot = top25_growth[["country", "iso_code"] + growth_cols].copy()

# Total NET growth per country
df_plot["total_growth"] = df_plot[growth_cols].sum(axis=1)

# Right edge of the bar (positive values only)
df_plot["positive_growth"] = df_plot[growth_cols].clip(lower=0).sum(axis=1)

# Sort DESCENDING
df_plot = df_plot.sort_values("total_growth", ascending=False)

# Add Flags to the country name using the updated manual mapping
df_plot["country_label"] = df_plot.apply(
    lambda row: f"{row['country']} {iso3_to_flag.get(row['iso_code'], row['iso_code'])}", 
    axis=1
)

# Long format
df_long = df_plot.melt(
    id_vars=["country_label", "total_growth"],
    value_vars=growth_cols,
    var_name="source",
    value_name="growth"
).round(2)

# Clean names
df_long["source"] = (
    df_long["source"]
    .str.replace("_elec_per_capita_growth", "", regex=False)
    .str.replace("_growth", "", regex=False)
    .str.replace("_", " ")
    .str.title()
)

df_long["source"] = df_long["source"].replace({
    "Other Renewables Elec Per Capita Exc Biofuel": "Other-renov"
})

# ---------------------------------------------------------
# 11. COLOR PALETTE (renewables)
# ---------------------------------------------------------
color_map = {
    "Solar": "#f1c40f",
    "Wind": "#2ca02c",
    "Hydro": "#1f77b4",
    "Biofuel": "#8c564b",
    "Other-renov": "#17becf"
}

# ---------------------------------------------------------
# 12. Horizontal stacked bar chart
# ---------------------------------------------------------
fig = px.bar(
    df_long,
    y="country_label",
    x="growth",
    color="source",
    orientation="h",
    color_discrete_map=color_map,
    title="Renewable Electricity Growth by Source (2000–2024)",
    labels={
        "growth": "Growth (kWh per capita)",
        "country_label": "Country",
        "source": "Renewable Source"
    },
    category_orders={"country_label": df_plot["country_label"].tolist()}
)

# ---------------------------------------------------------
# 13. PREMIUM LAYOUT
# ---------------------------------------------------------
fig.update_layout(
    barmode="relative",
    height=850,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    title=dict(
        text="Renewable Electricity Growth by Source",
        x=0.5, xanchor="center",
        font=dict(size=24, color="#383838")
    ),
    font=dict(color="#383838", size=14),
    xaxis=dict(
        title=dict(text="Growth (kWh per capita)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title=dict(text="Country", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838")
    ),
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom", y=1.05,
        xanchor="right", x=1,
        font=dict(size=14, color="#383838")
    ),
    margin=dict(l=80, r=40, t=80, b=40)
)

# ---------------------------------------------------------
# 14. Add total labels at the right edge
# ---------------------------------------------------------
for label, net_total, right_edge in zip(df_plot["country_label"], df_plot["total_growth"], df_plot["positive_growth"]):
    fig.add_annotation(
        x=right_edge,        
        y=label,
        text=f"  {net_total:.0f}", 
        showarrow=False,
        font=dict(size=12, color="#383838"),
        xanchor="left",
        yanchor="middle"
    )

# Run
fig.show()

# Save 
fig.write_html(
    r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\11_countries_growth_ren.html")

<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
Breaking Down the Renewable Boom<br>
</span>

<span style="font-size:18px;">
This visualization deconstructs the per‑capita growth of renewable electricity generation from 2000 to 2024.
</span>
<br><br>
<span style="font-size:22px; font-weight:700;">
Key Analytical Insights
</span>
<span style="font-size:18px;">
<br>
<b>• Diversified Growth:</b> While the overall net growth (annotated at the end of each bar) establishes the global ranking,<br> the color segmentation reveals the specific technological drivers for each nation—for example, the dominance of Wind and Solar across Northern Europe.

<b>• Negative Components:</b> The relative bar mode highlights structural shifts within national energy systems. Negative segments extending leftward (such as declining Hydro output in certain regions)<br>show that scaling modern green technologies like Solar and Wind is often required simply to compensate for the decline or retirement of older legacy renewable infrastructures.

</span>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Who Is Leading Fossil Energy Production per Capita?
</h2

In [208]:
# ---------------------------------------------------------
# 1. Filter valid countries (non-null ISO codes)
# ---------------------------------------------------------
df_countries = owid_completed[owid_completed["iso_code"].notna()]

# ---------------------------------------------------------
# 2. Select the analysis period (2000–2024)
# ---------------------------------------------------------
df_period = df_countries[df_countries["year"].between(2000, 2024)]

# ---------------------------------------------------------
# 3. Define fossil electricity breakdown columns (per capita)
# ---------------------------------------------------------
# Metrics represent electricity generation in kilowatt-hours (kWh) per person [4-6]
fossil_breakdown = [
    "coal_elec_per_capita",
    "oil_elec_per_capita",
    "gas_elec_per_capita"]

# ---------------------------------------------------------
# 4. Extract data snapshots for years 2000 and 2024
# ---------------------------------------------------------
# AGGIUNTO "iso_code" QUI ↓
base_cols = ["country", "iso_code", "fossil_elec_per_capita"] + fossil_breakdown

df_2000 = (
    df_period[df_period["year"] == 2000][base_cols]
    .rename(columns={"fossil_elec_per_capita": "fos_2000"}))

df_2024 = (
    df_period[df_period["year"] == 2024][base_cols]
    .rename(columns={"fossil_elec_per_capita": "fos_2024"}))

# ---------------------------------------------------------
# 5. Merge both years and recalculate consistent totals
# ---------------------------------------------------------
# AGGIUNTO "iso_code" COME CHIAVE DI MERGE 
df_fossil_growth = pd.merge(df_2000, df_2024, on=["country", "iso_code"], how="inner")

# We recalculate the totals by summing the components to ensure visual consistency in the chart [7]
df_fossil_growth["fos_2000_consistent"] = df_fossil_growth[[f"{c}_x" for c in fossil_breakdown]].sum(axis=1)
df_fossil_growth["fos_2024_consistent"] = df_fossil_growth[[f"{c}_y" for c in fossil_breakdown]].sum(axis=1)

# Compute net growth (2024 total - 2000 total)
df_fossil_growth["net_fossil_growth"] = df_fossil_growth["fos_2024_consistent"] - df_fossil_growth["fos_2000_consistent"]

# Compute growth for each specific fossil source (allows for negative values)
for col in fossil_breakdown:
    df_fossil_growth[f"{col}_growth"] = df_fossil_growth[f"{col}_y"] - df_fossil_growth[f"{col}_x"]

# ---------------------------------------------------------
# 6. Select Top 25 countries by highest net fossil growth
# ---------------------------------------------------------
top25_fossil = df_fossil_growth.sort_values("net_fossil_growth", ascending=False).head(25)

top25_fossil.head(10)

,country,iso_code,fos_2000,coal_elec_per_capita_x,oil_elec_per_capita_x,gas_elec_per_capita_x,fos_2024,coal_elec_per_capita_y,oil_elec_per_capita_y,gas_elec_per_capita_y,fos_2000_consistent,fos_2024_consistent,net_fossil_growth,coal_elec_per_capita_growth,oil_elec_per_capita_growth,gas_elec_per_capita_growth
159,Qatar,QAT,14155.192383,0.000000,0.000000,14155.192383,19190.212891,0.000000,0.000000,19190.212891,14155.192383,19190.212891,5035.020508,0.000000,0.000000,5035.020508
171,Saudi Arabia,SAU,8572.894531,0.000000,4626.729980,3946.164795,13097.290039,0.000000,4618.000488,8479.290039,8572.894775,13097.290527,4524.395752,0.000000,-8.729492,4533.125244
190,Taiwan,TWN,6301.473145,4039.336914,1441.820801,820.315430,10548.392578,4980.623047,183.510239,5384.259277,6301.473145,10548.392563,4246.919418,941.286133,-1258.310562,4563.943848
28,Brunei,BRN,7781.171387,0.000000,61.269066,7719.902344,12061.890625,2615.645508,105.645023,9295.009180,7781.171410,12016.299710,4235.128300,2615.645508,44.375957,1575.106836
148,Oman,OMN,3991.868896,0.000000,683.569214,3308.299805,7863.233887,0.000000,225.313004,7637.920898,3991.869019,7863.233902,3871.364883,0.000000,-458.256210,4329.621094
40,China,CHN,876.903381,835.125793,37.232750,4.544806,4394.495117,4105.906250,62.621479,225.967163,876.903349,4394.494892,3517.591543,3270.780457,25.388729,221.422357
182,South Korea,KOR,3792.658936,2456.450684,701.995850,634.212463,7268.514160,3688.880615,131.096603,3448.536865,3792.658997,7268.514084,3475.855087,1232.429932,-570.899246,2814.324402
199,Turkmenistan,TKM,2149.399658,0.000000,0.000000,2149.399658,5030.470386,0.000000,0.000000,5030.470386,2149.399658,5030.470386,2881.070728,0.000000,0.000000,2881.070728
77,Gibraltar,GIB,4687.387207,0.000000,4687.387207,0.000000,5850.144043,0.000000,0.000000,7241.291943,4687.387207,7241.291943,2553.904736,0.000000,-4687.387207,7241.291943
167,Saint Pierre and Miquelon,SPM,6332.119629,0.000000,6332.119629,0.000000,8807.485156,0.000000,8807.485156,0.000000,6332.119629,8807.485156,2475.365527,0.000000,2475.365527,0.000000


In [209]:
# ---------------------------------------------------------
# 0. MANUAL ISO TO FLAG MAPPING
# ---------------------------------------------------------
iso3_to_flag = {
    "QAT": "🇶🇦", "SAU": "🇸🇦", "TWN": "🇹🇼", "BRN": "🇧🇳", "OMN": "🇴🇲",
    "CHN": "🇨🇳", "KOR": "🇰🇷", "TKM": "🇹🇲", "GIB": "🇬🇮", "SPM": "🇵🇲",
    "IRQ": "🇮🇶", "IRN": "🇮🇷", "TTO": "🇹🇹", "BHR": "🇧🇭", "NCL": "🇳🇨",
    "KAZ": "🇰🇿", "SGP": "🇸🇬", "MYS": "🇲🇾", "BIH": "🇧🇦", "SYC": "🇸🇨",
    "FRO": "🇫🇴", "LBY": "🇱🇾", "VNM": "🇻🇳", "ATG": "🇦🇬", "LAO": "🇱🇦"
}

# ---------------------------------------------------------
# 7. Prepare data for horizontal stacked bar chart
# ---------------------------------------------------------
fossil_growth_cols = [f"{c}_growth" for c in fossil_breakdown]

df_plot = top25_fossil[["country", "iso_code"] + fossil_growth_cols].copy()

# Total NET growth per country
df_plot["total_growth"] = df_plot[fossil_growth_cols].sum(axis=1)

# Right edge of the bar (sum of positive values only) to avoid label overlapping
df_plot["positive_growth"] = df_plot[fossil_growth_cols].clip(lower=0).sum(axis=1)

# Sort DESCENDING → biggest on top
df_plot = df_plot.sort_values("total_growth", ascending=False)

# Add Flags to the country name using the manual mapping
df_plot["country_label"] = df_plot.apply(
    lambda row: f"{row['country']} {iso3_to_flag.get(row['iso_code'], row['iso_code'])}", 
    axis=1
)

# Long format
df_long_fossil = df_plot.melt(
    id_vars=["country_label", "total_growth"],
    value_vars=fossil_growth_cols,
    var_name="source",
    value_name="growth"
)

# Clean names
df_long_fossil["source"] = (
    df_long_fossil["source"]
    .str.replace("_elec_per_capita_growth", "", regex=False)
    .str.replace("_growth", "", regex=False)
    .str.replace("_", " ")
    .str.title()
)

# ---------------------------------------------------------
# 8. COLOR PALETTE (fossil)
# ---------------------------------------------------------
color_map_fossil = {
    "Coal": "#4c4c4c",     # dark gray
    "Oil": "#b5651d",      # brown
    "Gas": "#C39BD3"       # light purple
}

# ---------------------------------------------------------
# 9. Create the horizontal stacked bar chart
# ---------------------------------------------------------
fig = px.bar(
    df_long_fossil,
    y="country_label",   
    x="growth",
    color="source",
    orientation="h",
    color_discrete_map=color_map_fossil,
    title="Fossil Electricity Growth by Source (Top 25)",
    labels={
        "growth": "Growth (kWh per capita)",
        "country_label": "Country",
        "source": "Fossil Source"
    },
    category_orders={"country_label": df_plot["country_label"].tolist()}
)

# ---------------------------------------------------------
# 10. PREMIUM LAYOUT
# ---------------------------------------------------------
fig.update_layout(
    barmode="relative",
    height=850,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    title=dict(
        text="Fossil Electricity Growth by Source (Top 25)",
        x=0.5, xanchor="center",
        font=dict(size=24, color="#383838")
    ),
    font=dict(color="#383838", size=14),
    xaxis=dict(
        title=dict(text="Growth (kWh per capita)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title=dict(text="Country", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838")
    ),
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom", y=1.05,
        xanchor="right", x=1,
        font=dict(size=14, color="#383838")
    ),
    margin=dict(l=80, r=40, t=80, b=40)
)

# ---------------------------------------------------------
# 11. Add total labels at the end of each stacked bar
# ---------------------------------------------------------
for label, net_total, right_edge in zip(df_plot["country_label"], df_plot["total_growth"], df_plot["positive_growth"]):
    fig.add_annotation(
        x=right_edge,
        y=label,
        text=f"  {net_total:.0f}", 
        showarrow=False,
        font=dict(size=12, color="#383838"),
        xanchor="left",
        yanchor="middle"
    )

# Run
fig.show()

# Export HTML
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\12_countries_growth_fossil.html")

<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
Fossil Fuels: The Lingering Legacy<br>
</span>
<span style="font-size:18px;">
This visualization serves as the direct counterpart to the renewable growth analysis, focusing on the expansion (and contraction) of fossil fuel dependency from 2000 to 2024.<br><br>
</span>
<span style="font-size:22px; font-weight:700;">
Key Analytical Insights
</span>
<br>
<span style="font-size:18px;">
<b>• Persistent Growth:</b> Net positive values on the right reveal the nations still expanding their high‑carbon energy systems, often driven by rapid industrialization and rising energy demand.<br>
<b>• The Phase‑Out (Negative Growth):</b> The relative bar mode is essential here. Segments extending leftward (negative growth) represent the structural phase‑out of legacy fossil sources.<br> A large negative Coal or Oil bar is a strong indicator of an electricity grid undergoing active and successful decarbonization.

</span>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Countries Leading the Net Energy Transition per Capita (2000–2024)<br>
"renewables_energy_per_capita" VS "fossil_energy_per_capita"
</h2



In [210]:
# 1. Filter real countries
df_countries = owid_completed[owid_completed["iso_code"].notna()]

# 2. Select the period 2000–2024
df_period = df_countries[df_countries["year"].between(2000, 2024)]

# 3. Keep only rows with both metrics
df_period = df_period[
    df_period["renewables_energy_per_capita"].notna() &
    df_period["fossil_energy_per_capita"].notna()]

# 4. Aggregate per country (sum over years)
df_gap = (
    df_period.groupby("country")[["renewables_energy_per_capita", "fossil_energy_per_capita"]]
    .sum().round(2)
    .reset_index())

# 5. Compute the GAP
df_gap["gap_per_capita"] = (
    df_gap["renewables_energy_per_capita"] - df_gap["fossil_energy_per_capita"])

# 6. Sort by GAP (descending: best transitions first)
df_gap_sorted = df_gap.sort_values(by="gap_per_capita", ascending=False)

df_gap_sorted.head(25)

,country,renewables_energy_per_capita,fossil_energy_per_capita,gap_per_capita
27,Iceland,3158505.56,838476.41,2320029.15
47,Norway,1758300.55,825912.88,932387.67
65,Sweden,628244.71,540972.26,87272.45
5,Bangladesh,443.48,49346.93,-48903.45
8,Brazil,166135.70,220654.58,-54518.88
64,Sri Lanka,15858.17,77413.39,-61555.22
49,Pakistan,9933.73,81362.13,-71428.40
51,Philippines,15837.30,88670.54,-72833.24
50,Peru,54968.92,141937.78,-86968.86
13,Colombia,69786.52,175339.95,-105553.43


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Countries Leading the Net Energy Transition per Capita (2000–2024)<br>
"low_carbon_energy_per_capita" VS "fossil_energy_per_capita"]
</h2


In [211]:
# 1. Filter real countries
df_countries = owid_completed[owid_completed["iso_code"].notna()]

# 2. Select the period 2000–2024
df_period = df_countries[df_countries["year"].between(2000, 2024)]

# 3. Keep only rows with both metrics
df_period = df_period[
    df_period["low_carbon_energy_per_capita"].notna() &
    df_period["fossil_energy_per_capita"].notna()]

# 4. Aggregate per country (sum over years)
df_low = (
    df_period.groupby("country")[["low_carbon_energy_per_capita", "fossil_energy_per_capita"]]
    .sum().round(2)
    .reset_index())

# 5. Compute the GAP low‑carbon – fossils
df_low["gap_lowcarbon_per_capita"] = (
    df_low["low_carbon_energy_per_capita"] - df_low["fossil_energy_per_capita"])

# 6. Sort by GAP (descending: best transitions first)
df_low_sorted = df_low.sort_values(by="gap_lowcarbon_per_capita", ascending=False)

df_low_sorted.head(30)


,country,low_carbon_energy_per_capita,fossil_energy_per_capita,gap_lowcarbon_per_capita
27,Iceland,3158505.56,838476.41,2320029.15
47,Norway,1758300.55,825912.88,932387.67
65,Sweden,1050728.01,540972.26,509755.75
66,Switzerland,522194.78,526950.21,-4755.43
5,Bangladesh,443.48,49346.93,-48903.45
8,Brazil,170748.21,220654.58,-49906.37
64,Sri Lanka,15858.17,77413.39,-61555.22
49,Pakistan,11809.87,81362.13,-69552.26
51,Philippines,15837.30,88670.54,-72833.24
22,France,520979.47,595481.26,-74501.79


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
    Comparing Energy Transition Metrics per Capita
</h2>

<h3 style="text-align:center; color:grey; font-size:18px; line-height:1.5;">

<span style="color:gold; font-weight:bold;">1. Renewables – Fossils per Capita</span><br>
Measures the net contribution of renewable energy sources relative to fossil fuels.<br><br>

<span style="color:gold; font-weight:bold;">2. Low‑Carbon – Fossils per Capita</span><br>
Includes nuclear energy together with renewables, offering a more complete picture of decarbonization.<br>
Placing both charts side by side reveals how the inclusion of nuclear energy reshapes the ranking<br>
and highlights countries that rely heavily on low‑carbon electricity systems.

</h3>



In [212]:
# ---------------------------------------------------------
# 0. FINAL UPDATED ISO TO FLAG MAPPING
# ---------------------------------------------------------
iso3_to_flag = {
    "QAT": "🇶🇦", "SAU": "🇸🇦", "TWN": "🇹🇼", "BRN": "🇧🇳", "OMN": "🇴🇲",
    "CHN": "🇨🇳", "KOR": "🇰🇷", "TKM": "🇹🇲", "GIB": "🇬🇮", "SPM": "🇵🇲",
    "IRQ": "🇮🇶", "IRN": "🇮🇷", "TTO": "🇹🇹", "BHR": "🇧🇭", "NCL": "🇳🇨",
    "KAZ": "🇰🇿", "SGP": "🇸🇬", "MYS": "🇲🇾", "BIH": "🇧🇦", "SYC": "🇸🇨",
    "FRO": "🇫🇴", "LBY": "🇱🇾", "VNM": "🇻🇳", "ATG": "🇦🇬", "LAO": "🇱🇦",
    "ISL": "🇮🇸", "BTN": "🇧🇹", "GRL": "🇬🇱", "DNK": "🇩🇰", "FIN": "🇫🇮",
    "NLD": "🇳🇱", "DEU": "🇩🇪", "AUS": "🇦🇺", "URY": "🇺🇾", "PRT": "🇵🇹",
    "ESP": "🇪🇸", "EST": "🇪🇪", "GRC": "🇬🇷", "IRL": "🇮🇪", "BEL": "🇧🇪",
    "ALB": "🇦🇱", "LTU": "🇱🇹", "AUT": "🇦🇹", "GBR": "🇬🇧", "SWE": "🇸🇪",
    "CHL": "🇨🇱", "USA": "🇺🇸", "NOR": "🇳🇴", "BGD": "🇧🇩", "BRA": "🇧🇷", 
    "LKA": "🇱🇰", "PAK": "🇵🇰", "PHL": "🇵🇭", "PER": "🇵🇪", "COL": "🇨🇴", 
    "IND": "🇮🇳", "MAR": "🇲🇦", "ECU": "🇪🇨", "IDN": "🇮🇩", "CHE": "🇨🇭", 
    "EGY": "🇪🇬", "MKD": "🇲🇰", "LVA": "🇱🇻", "ROU": "🇷🇴", "TUR": "🇹🇷", 
    "DZA": "🇩🇿", "NZL": "🇳🇿", "FRA": "🇫🇷", "SVN": "🇸🇮"
}

# =========================
# 1. FILTER DATA
# =========================
df = owid_completed.copy()
df = df[df["iso_code"].notna()]
df = df[
    df["renewables_energy_per_capita"].notna() &
    df["low_carbon_energy_per_capita"].notna() &
    df["fossil_energy_per_capita"].notna()
]

# Absolute gap per capita
df["gap_renewables_per_capita"] = df["renewables_energy_per_capita"] - df["fossil_energy_per_capita"]
df["gap_lowcarbon_per_capita"] = df["low_carbon_energy_per_capita"] - df["fossil_energy_per_capita"]

# =========================
# 2. AGGREGATED PERIODS
# =========================
start_years = [2000, 2005, 2010, 2015, 2020]
frames = []

for start in start_years:
    df_period = df[df["year"].between(start, 2024)]
    df_agg = df_period.groupby(["iso_code", "country"])[
        ["gap_renewables_per_capita", "gap_lowcarbon_per_capita"]
    ].mean().reset_index()
    df_agg["period"] = f"{start} - 2024"
    frames.append(df_agg)

df_anim = pd.concat(frames, ignore_index=True)

# Add Flags to the country label
df_anim["country_label"] = df_anim.apply(
    lambda row: f"{row['country']} {iso3_to_flag.get(row['iso_code'], row['iso_code'])}", 
    axis=1
)

# =========================
# 3. TOP 25 PER PERIODO
# =========================
df_ren = (
    df_anim.sort_values(["period", "gap_renewables_per_capita"], ascending=[True, False])
    .groupby("period").head(25)
)
df_ren = df_ren.sort_values(by=["period", "gap_renewables_per_capita"], ascending=[True, True])

df_low = (
    df_anim.sort_values(["period", "gap_lowcarbon_per_capita"], ascending=[True, False])
    .groupby("period").head(25)
)
df_low = df_low.sort_values(by=["period", "gap_lowcarbon_per_capita"], ascending=[True, True])

# =========================
# 4. PREMIUM COLOR SCALES
# =========================
color_scale_ren = ["#8b0000", "#ff4d4d", "#32cd32", "#006400"]  
color_scale_low = ["#8b0000", "#ff4d4d", "#1e90ff", "#00008b"]  

# =========================
# 5. GRAPH 1 — RENEWABLES AGGREGATED GAP
# =========================
fig_ren = px.bar(
    df_ren,
    x="gap_renewables_per_capita",
    y="country_label",
    orientation="h",
    animation_frame="period",
    animation_group="country_label",
    color="gap_renewables_per_capita",
    color_continuous_scale=color_scale_ren,
    color_continuous_midpoint=0,    
    title="Aggregated Average Gap: Renewables vs Fossils (Top 25)",
    labels={"gap_renewables_per_capita": "Renewables Gap (kWh)", "country_label": "Country"}
)

fig_ren.update_layout(
    height=850, paper_bgcolor="#FFF8E7", plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    title=dict(text="Aggregated Average Gap: Renewables vs Fossils (Top 25)", x=0.5, xanchor="center", font=dict(size=24, color="#383838")),
    yaxis=dict(categoryorder="total ascending"),
    xaxis=dict(
        title=dict(text="Average Gap per Capita (kWh)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"), gridcolor="rgba(0,0,0,0.08)",
        range=[df_ren["gap_renewables_per_capita"].min() * 1.1, df_ren["gap_renewables_per_capita"].max() * 1.1]
    ),
    coloraxis_colorbar=dict(title=dict(text="Average<br>Gap", font=dict(color="#383838", size=14)), tickfont=dict(color="#383838")),
    margin=dict(l=80, r=40, t=80, b=40)
)

# =========================
# 6. GRAPH 2 — LOW-CARBON AGGREGATED GAP
# =========================
fig_low = px.bar(
    df_low,
    x="gap_lowcarbon_per_capita",
    y="country_label",
    orientation="h",
    animation_frame="period",
    animation_group="country_label",
    color="gap_lowcarbon_per_capita",
    color_continuous_scale=color_scale_low,
    color_continuous_midpoint=0,     
    title="Aggregated Average Gap: Low-Carbon vs Fossils (Top 25)",
    labels={"gap_lowcarbon_per_capita": "Low-Carbon Gap (kWh)", "country_label": "Country"}
)

fig_low.update_layout(
    height=850, paper_bgcolor="#FFF8E7", plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    title=dict(text="Aggregated Average Gap: Low-Carbon vs Fossils (Top 25)", x=0.5, xanchor="center", font=dict(size=24, color="#383838")),
    yaxis=dict(categoryorder="total ascending"),
    xaxis=dict(
        title=dict(text="Average Gap per Capita (kWh)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"), gridcolor="rgba(0,0,0,0.08)",
        range=[df_low["gap_lowcarbon_per_capita"].min() * 1.1, df_low["gap_lowcarbon_per_capita"].max() * 1.1]
    ),
    coloraxis_colorbar=dict(title=dict(text="Average<br>Gap", font=dict(color="#383838", size=14)), tickfont=dict(color="#383838")),
    margin=dict(l=80, r=40, t=80, b=40)
)

fig_ren.show()
fig_low.show()

# =========================
# 7. EXPORT (READY)
# =========================
fig_ren.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\13_renewables_gap.html")
fig_low.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\14_lowcarbon_gap.html")


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Clean Energy Deficit: Mind the Gap<br>
</span>

<br>

<span style="font-size:18px;">
These twin charts expose the stark reality of the global energy mix by calculating the per-capita "Gap" between clean energy and fossil fuel consumption. A bar extending to the right (green or blue) indicates a true clean energy surplus, while a bar pointing left (brown or purple) reveals a fossil fuel deficit.<br><br>
The top chart (Renewables vs. Fossils) highlights just how rare a true renewable surplus is, with only unique geographic outliers like Iceland and Norway achieving significant positive gaps. Almost every other nation operates at a deep fossil deficit.<br><br>
The bottom chart broadens the "clean" definition to include Nuclear power (Low-Carbon vs. Fossils). While this addition pushes a few more technologically advanced nations (like Sweden and Switzerland) across the zero line into surplus territory, the overwhelming visual takeaway across all analyzed time periods remains the persistent, deep-seated fossil fuel dependence of the global economy.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:100%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
Nuclear Efficiency and Its Role in the Green Transition
</h2>
<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:100%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

In [213]:
# =====================================================================
# 1. CLEANING & FILTERING
# =====================================================================
df_time = owid_completed[
    (owid_completed["iso_code"].notna()) &
    (owid_completed["year"].between(2000, 2024)) &
    (owid_completed["nuclear_share_elec"] > 0)
].copy()

# 2. ECONOMIC GAIN CALCULATION (Energy Productivity)
df_time = df_time.dropna(subset=["energy_per_gdp"])
df_time = df_time[df_time["energy_per_gdp"] > 0]
df_time["economic_gain"] = (1 / df_time["energy_per_gdp"]).round(2)

# Fundamental: order by year
df_time["year"] = df_time["year"].astype(int)
df_time = df_time.sort_values(by=["year", "country"]).dropna(subset=["nuclear_share_elec", "carbon_intensity_elec"])

# ---------------------------------------------------------
# 🌟 UPDATED: FLAG MAP FOR KEY COUNTRIES
# ---------------------------------------------------------
flag_dict = {
    "France": "🇫🇷", "Germany": "🇩🇪", "Switzerland": "🇨🇭", "Slovenia": "🇸🇮",
    "Sweden": "🇸🇪", "United States": "🇺🇸", "China": "🇨🇳", "Japan": "🇯🇵",
    "South Korea": "🇰🇷", "United Kingdom": "🇬🇧", "Spain": "🇪🇸", "Russia": "🇷🇺",
    "Belgium": "🇧🇪", "Brazil": "🇧🇷", "Argentina": "🇦🇷", "Ukraine": "🇺🇦",
    "Slovakia": "🇸🇰", "Hungary": "🇭🇺", "Finland": "🇫🇮", "Bulgaria": "🇧🇬",
    "Czechia": "🇨🇿", "Belarus": "🇧🇾", "Armenia": "🇦🇲", "Romania": "🇷🇴",
    "United Arab Emirates": "🇦🇪", "Canada": "🇨🇦", "Pakistan": "🇵🇰",
    "Netherlands": "🇳🇱", "Mexico": "🇲🇽", "Taiwan": "🇹🇼", "India": "🇮🇳",
    "Iran": "🇮🇷", "South Africa": "🇿🇦"
}

# Create the 'flag' column
df_time["flag"] = df_time["country"].map(flag_dict).fillna("")
# ---------------------------------------------------------

# 3. SCALE PREPARATION
y_range = [0, df_time["carbon_intensity_elec"].max() * 1.1]
x_range = [0, df_time["nuclear_share_elec"].max() * 1.1]

# 4. ANIMATED VISUALIZATION
fig = px.scatter(
    df_time,
    x="nuclear_share_elec",
    y="carbon_intensity_elec",
    animation_frame="year",
    animation_group="country",
    size="economic_gain",
    size_max=50,
    color="economic_gain",
    trendline="ols",
    color_continuous_scale="viridis",
    hover_name="country",
    text="flag",  # Using flag column for labels
    
    hover_data={"gdp": True, "population": True, "economic_gain": True, "year": False, "flag": False},
    range_x=x_range,
    range_y=y_range
)

fig.update_layout(
    height=800,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    
    title=dict(
        text="The Nuclear Effect: Nuclear Share vs. Carbon Intensity",
        x=0.5,
        xanchor="center",
        font=dict(size=24, color="#383838")
    ),

    xaxis=dict(
        title=dict(text="Nuclear Share in Electricity Mix (%)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title=dict(text="Carbon Intensity (gCO₂e per kWh)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838"),
        gridcolor="rgba(0,0,0,0.08)"
    ),

    coloraxis_colorbar=dict(
        title=dict(text="Economic Gain<br>($ per kWh)", font=dict(color="#383838", size=14)),
        tickfont=dict(color="#383838")
    ),
    margin=dict(l=80, r=40, t=80, b=40)
)

# Trendline styling
fig.update_traces(
    line=dict(color="#383838", dash="dash", width=2),
    selector=dict(mode="lines")
)

# Bubble and Text (Flags) styling
fig.update_traces(
    marker=dict(line=dict(width=1, color="black"), opacity=0.8),
    textposition='top center',
    textfont=dict(size=18),
    selector=dict(mode="markers")
)

fig.show()

# Final export
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\15_nuclear_scatter.html")


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Nuclear Effect: Share vs. Carbon Intensity
</span>
<br>
<span style="font-size:18px;">
This visualization isolates the impact of atomic energy on grid decarbonization, plotting the <i>Nuclear Share in the Electricity Mix</i> (X‑axis) against overall <i>Carbon Intensity</i> (Y‑axis).
</span>
<br>
<br>
<span style="font-size:22px; font-weight:700;">
Key Analytical Insights
</span>
<br>
<span style="font-size:18px;">
<b>• The Decarbonization Trend:</b>
The dashed OLS trendline reveals a clear and robust negative correlation.<br>
Countries with a high nuclear share (moving right) consistently achieve much lower carbon intensity (moving downward), demonstrating nuclear energy’s strong effectiveness in rapid grid decarbonization.
<br>
<br>
<b>• The Phase‑Out Phenomenon (Disappearing Nations):</b>
A unique aspect of the animation is that some bubbles vanish over time. This visually represents political nuclear phase‑outs (e.g., Germany).<br>
As these nations decommission reactors and their nuclear share falls to zero, they disappear from the nuclear landscape entirely.
<br>

<b>• Economic Efficiency:</b>The color scale (<i>Economic Gain</i>) adds a third analytical layer, showing that high nuclear dependency does not hinder economic performance.<br>Several high‑nuclear countries maintain strong structural efficiency while achieving deep decarbonization.
</span>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:gold; font-size:28px;">
    CAPACITY INSTALLED
</h2>
<h2 style="text-align:center; color:gold; font-size:28px;">
    import OWID merge with Irena
</h2>


In [214]:
# Import owid(world_bank) merge con Irena

path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Owid_WB_Irena.csv"
Owid_WB_Irena = pd.read_csv(path)
Owid_WB_Irena.head()

,country,year,iso_code,population,biofuel_share_energy,coal_share_energy,electricity_share_energy,fossil_share_energy,gas_share_energy,hydro_share_energy,...,hydro_capacity_gw,nuclear_capacity_gw,oil_capacity_gw,other_fossil_capacity_gw,other_renewables_capacity_gw,solar_capacity_gw,wind_capacity_gw,fossil_capacity_gw,renewables_capacity_gw,low_carbon_capacity_gw
0,Afghanistan,2000,AFG,20130334.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.191503,0.0,0.002798,0.026927,0.0,0.0,0.0,0.029725,0.191503,0.191503
1,Afghanistan,2001,AFG,20284303.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.191503,0.0,0.002798,0.026927,0.0,0.0,0.0,0.029725,0.191503,0.191503
2,Afghanistan,2002,AFG,21378123.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.191506,0.0,0.002798,0.026927,0.0,0.0,0.0,0.029725,0.191506,0.191506
3,Afghanistan,2003,AFG,22733053.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.191619,0.0,0.010106,0.026927,0.0,0.0,0.0,0.037033,0.191619,0.191619
4,Afghanistan,2004,AFG,23560656.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.191718,0.0,0.025086,0.026927,0.0,0.0,0.0,0.052013,0.191718,0.191718


In [215]:
# 1. FILTER DATA
# Select recent years (2015–2024) and valid ISO-coded countries
df_cap = Owid_WB_Irena[
    (Owid_WB_Irena["year"].between(2015, 2024)) &
    (Owid_WB_Irena["iso_code"].notna())
].copy()

# 2. CALCULATE REAL YIELD (TWh generated per installed GW)
df_cap["yield_solar"]   = np.where(df_cap["solar_capacity_gw"] > 1,   df_cap["solar_electricity"]   / df_cap["solar_capacity_gw"],   np.nan)
df_cap["yield_wind"]    = np.where(df_cap["wind_capacity_gw"] > 1,    df_cap["wind_electricity"]    / df_cap["wind_capacity_gw"],    np.nan)
df_cap["yield_hydro"]   = np.where(df_cap["hydro_capacity_gw"] > 1,   df_cap["hydro_electricity"]   / df_cap["hydro_capacity_gw"],   np.nan)
df_cap["yield_nuclear"] = np.where(df_cap["nuclear_capacity_gw"] > 1, df_cap["nuclear_electricity"] / df_cap["nuclear_capacity_gw"], np.nan)
df_cap["yield_gas"]     = np.where(df_cap["gas_capacity_gw"] > 1,     df_cap["gas_electricity"]     / df_cap["gas_capacity_gw"],     np.nan)
df_cap["yield_oil"]     = np.where(df_cap["oil_capacity_gw"] > 1,     df_cap["oil_electricity"]     / df_cap["oil_capacity_gw"],     np.nan)
df_cap["yield_coal"]    = np.where(df_cap["coal_capacity_gw"] > 1,    df_cap["coal_electricity"]    / df_cap["coal_capacity_gw"],    np.nan)

# 3. TRANSFORM INTO LONG FORMAT FOR PLOTLY
df_melted = df_cap.melt(
    id_vars=["country", "year"],
    value_vars=[
        "yield_solar", "yield_wind", "yield_hydro", 
        "yield_nuclear", "yield_gas", "yield_oil", "yield_coal"
    ],
    var_name="technology",
    value_name="twh_per_gw"
)

# Clean technology labels
df_melted["technology"] = df_melted["technology"].replace({
    "yield_solar": "Solar",
    "yield_wind": "Wind",
    "yield_hydro": "Hydro",
    "yield_nuclear": "Nuclear",
    "yield_gas": "Natural Gas",
    "yield_oil": "Oil",
    "yield_coal": "Coal"
})

# Drop NaN
df_melted = df_melted.dropna(subset=["twh_per_gw"])

# ---------------------------------------------------------
# SCIENTIFIC FIX: The physical maximum limit is 8.76 TWh.
# We remove the dataset’s reporting errors!
# ---------------------------------------------------------
df_melted = df_melted[df_melted["twh_per_gw"] <= 9.0]

In [216]:
# Make sure the year is an integer for the menu labels
df_melted["year"] = df_melted["year"].astype(int)

# Map and order colors
techs = ["Solar", "Wind", "Hydro", "Nuclear", "Natural Gas", "Oil", "Coal"]
color_map = {
    "Solar": "#f1c40f",       
    "Wind": "#2ca02c",        
    "Hydro": "#00b4d8",       
    "Nuclear": "#4361ee",     
    "Natural Gas": "#C39BD3", 
    "Oil": "#8B4513",         
    "Coal": "#4c4c4c"         
}

# Find all years present in the dataset (e.g., from 2015 to 2024)
years = sorted(df_melted["year"].unique())

fig = go.Figure()

# --- STEP 4A: Create the traces for "TOTAL (2015–2024)" (Visible by default) ---
for tech in techs:
    df_tech = df_melted[df_melted["technology"] == tech]
    fig.add_trace(go.Box(
        x=df_tech["technology"],
        y=df_tech["twh_per_gw"],
        name=tech,
        marker_color=color_map[tech],
        boxpoints="all",
        jitter=0.4,
        pointpos=0, # Centra i punti sopra il boxplot
        customdata=np.stack((df_tech["country"], df_tech["year"]), axis=-1) if not df_tech.empty else None,
        hovertemplate="<b>%{customdata[0]}</b><br>Year: %{customdata[1]}<br>Yield: %{y:.2f} TWh/GW<extra></extra>",
        visible=True
    ))

# --- STEP 4B: Create the traces for the INDIVIDUAL YEARS (Hidden by default) ---
for year in years:
    df_year = df_melted[df_melted["year"] == year]
    for tech in techs:
        df_tech = df_year[df_year["technology"] == tech]
        
        
        if df_tech.empty:
            custom_data = None
            x_data = [tech]
            y_data = [None]
        else:
            custom_data = np.stack((df_tech["country"], df_tech["year"]), axis=-1)
            x_data = df_tech["technology"]
            y_data = df_tech["twh_per_gw"]
            
        fig.add_trace(go.Box(
            x=x_data,
            y=y_data,
            name=tech,
            marker_color=color_map[tech],
            boxpoints="all",
            jitter=0.4,
            pointpos=0,
            customdata=custom_data,
            hovertemplate="<b>%{customdata[0]}</b><br>Year: %{customdata[1]}<br>Yield: %{y:.2f} TWh/GW<extra></extra>",
            visible=False
        ))

# --- STEP 5: MENU DROPDOWN ---
n_techs = len(techs)
total_traces = n_techs * (len(years) + 1)
buttons = []


visible_total = [True] * n_techs + [False] * (total_traces - n_techs)
buttons.append(dict(
    label="All Years (2015-2024)",
    method="update",
    args=[{"visible": visible_total},
          {"title": "The Capacity Paradox: Real Energy Yield per Installed GW (2015–2024)"}]
))


for i, year in enumerate(years):
    visible_year = [False] * total_traces
    # Calcoliamo dove iniziano i 7 trace di questo specifico anno per accenderli
    start_idx = n_techs + (i * n_techs)
    for j in range(n_techs):
        visible_year[start_idx + j] = True
        
    buttons.append(dict(
        label=str(year),
        method="update",
        args=[{"visible": visible_year},
              {"title": f"The Capacity Paradox: Real Energy Yield per Installed GW ({year})"}]
    ))

# --- STEP 6: LAYOUT PREMIUM ---
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        x=0.01,
        xanchor="left",
        y=1.15,
        yanchor="top",
        showactive=True,
        bgcolor="#FFF8E7",      
        bordercolor="#383838",
        font=dict(size=14, color="#383838")
    )],
    
    height=800,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),

    title=dict(
        text="The Capacity Paradox: Real Energy Yield per Installed GW (2015–2024)",
        x=0.5,
        xanchor="center",
        y=0.95, # Alzato leggermente per fare spazio al menu
        font=dict(size=24, color="#383838")
    ),

    xaxis=dict(
        title=dict(text="Energy Source", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838", size=16),
        gridcolor="rgba(0,0,0,0)",
        categoryorder="array",
        categoryarray=techs
    ),
    
    yaxis=dict(
        title=dict(text="Real Yield (TWh generated per GW installed)", font=dict(size=16, color="#383838")),
        tickfont=dict(color="#383838", size=14),
        gridcolor="rgba(0,0,0,0.08)",
        range=[0, 9.5] 
    ),
    
    showlegend=False, 
    margin=dict(t=120) 
)

fig.show()
fig.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\16_nuclear_box.html")


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Capacity Paradox: Installed GW vs. Actual Generation<br>
</span>

<br>

<span style="font-size:18px;">
This box plot reveals a critical, yet often misunderstood aspect of the energy transition: the vast difference between a power plant's "installed capacity" (measured in GW) and its "actual energy generated" (measured in TWh). <i>Note: The theoretical physical maximum is 8.76 TWh per GW per year.</i><br><br>
The data clearly demonstrates that 1 GW of capacity is not created equal across technologies. Nuclear power operates reliably near its physical limit, acting as a stable baseload. In contrast, weather-dependent renewables like Solar and Wind inherently possess much lower real-world energy yields per installed unit.<br><br>
This highlights the "Capacity Paradox"—we cannot evaluate the energy transition merely by comparing the GW of new solar/wind additions directly to the GW of retiring traditional plants. To replace the actual electricity generated by a single GW of firm nuclear or fossil baseload, we must build several GWs of renewable capacity to compensate for their intermittency.
</span>

</div>


<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:100%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">
<h2 style="text-align:center; color:gold; font-size:28px;">
    Global map of energy transition categories
</h2>
<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:100%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

In [ ]:
# 1. Configuration
geojson_source = requests.get("https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json").json()
load_dotenv(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\.env")
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN")


# 2. Data
df_base = owid_completed[owid_completed["iso_code"].notna()].copy()
cons_check = df_base[df_base["year"].between(2019, 2024)].groupby("iso_code")[["low_carbon_consumption", "fossil_fuel_consumption"]].mean().sum(axis=1)
valid_isos = cons_check[cons_check > 0].index

df_ele = df_base[df_base["year"].between(2019, 2024)].groupby(["country", "iso_code"])[["low_carbon_elec_per_capita", "fossil_elec_per_capita"]].mean().reset_index()
df_ele["total"] = df_ele[["low_carbon_elec_per_capita", "fossil_elec_per_capita"]].sum(axis=1)
df_ele = df_ele[(df_ele["iso_code"].isin(valid_isos)) & (df_ele["total"] > 0)].copy()
df_ele["clean_pct"] = (df_ele["low_carbon_elec_per_capita"] / df_ele["total"]) * 100

def get_cat_eng(pct):
    if pct >= 50: return "Low-carbon > Fossil"
    elif pct >= 20: return "Balanced (low close to fossil)"
    elif pct >= 5: return "Low-carbon not sufficient"
    else: return "Fossil giant"

df_ele["category"] = df_ele["clean_pct"].apply(get_cat_eng)

# 3. GeoJSON Mapping (Copy)
ele_geojson = copy.deepcopy(geojson_source)
color_map = {
    "Low-carbon > Fossil": [212, 175, 55, 160], "Balanced (low close to fossil)": [135, 206, 250, 160],
    "Low-carbon not sufficient": [139, 26, 26, 160], "Fossil giant": [59, 0, 0, 160], "No data": [200, 200, 200, 100]
}

for f in ele_geojson["features"]:
    iso = f.get("properties", {}).get("ISO_A3") or f.get("id")
    match = df_ele[df_ele["iso_code"] == iso]
    if not match.empty:
        f["properties"]["category"] = match.iloc[0]["category"]
        f["properties"]["color"] = color_map[match.iloc[0]["category"]]
        f["properties"]["elevation"] = match.iloc[0]["total"]
    else:
        f["properties"]["category"] = "No data"
        f["properties"]["color"] = color_map["No data"]
        f["properties"]["elevation"] = 0

# 4. Render 
deck_ele = pdk.Deck(
    layers=[pdk.Layer("GeoJsonLayer", ele_geojson, extruded=True, pickable=True, filled=True, get_elevation="properties.elevation",
                       elevation_scale=10, get_fill_color="properties.color", get_line_color=[255, 255, 255, 100])],
    initial_view_state=pdk.ViewState(latitude=20, longitude=0, zoom=1.5, pitch=40),
    map_provider="mapbox",
    map_style="mapbox://styles/mapbox/light-v10", 
    api_keys={"mapbox": MAPBOX_TOKEN},
    tooltip={"html": "<b>{name}</b><br/>{category}", "style": {"backgroundColor": "white", "color": "black"}}
)

# Save
deck_ele.to_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\17_global_map_ele.html")
# Show
deck_ele.show()

<div style="
    text-align: center; 
    background: linear-gradient(145deg, #2a2a2a, #1a1a1a); 
    border: 1px solid #444; 
    padding: 30px; 
    border-radius: 16px; 
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    margin: 20px 0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
">
    <div style="color: #00d2ff; font-size: 24px; margin-bottom: 10px;">⚡</div>
    <h3 style="color: #ffffff; margin-top: 0; font-weight: 600;">Security Note: Interactive Electricity Map</h3>
    <p style="color: #cccccc; font-size: 1.1em; line-height: 1.6;">
        Due to security protocols regarding the Mapbox API token, 
        this interactive visualization is hosted externally.
    </p>
    <a href="./Plot/17_global_map_ele.html" target="_blank" style="
        display: inline-block; 
        margin-top: 15px; 
        padding: 12px 25px; 
        background: linear-gradient(to right, #3a7bd5, #00d2ff); 
        color: white; 
        text-decoration: none; 
        border-radius: 50px; 
        font-weight: bold;
        box-shadow: 0 4px 10px rgba(0, 210, 255, 0.3);
    ">
        🌍 Open Interactive Electricity Map
    </a>
</div>


In [ ]:
# Configuration
df_cons = df_base[df_base["year"].between(2019, 2024)].groupby(["country", "iso_code"])[["low_carbon_consumption", "fossil_fuel_consumption"]].mean().reset_index()
df_cons["total"] = df_cons[["low_carbon_consumption", "fossil_fuel_consumption"]].sum(axis=1)
df_cons = df_cons[(df_cons["iso_code"].isin(valid_isos)) & (df_cons["total"] > 0)].copy()
df_cons["clean_pct"] = (df_cons["low_carbon_consumption"] / df_cons["total"]) * 100
df_cons["category"] = df_cons["clean_pct"].apply(get_cat_eng)

# 3. GeoJSON Mapping (Copia pulita)
cons_geojson = copy.deepcopy(geojson_source)
for f in cons_geojson["features"]:
    iso = f.get("properties", {}).get("ISO_A3") or f.get("id")
    match = df_cons[df_cons["iso_code"] == iso]
    if not match.empty:
        f["properties"]["category"] = match.iloc[0]["category"]
        f["properties"]["color"] = color_map[match.iloc[0]["category"]]
        f["properties"]["elevation"] = match.iloc[0]["total"]
    else:
        f["properties"]["category"] = "No data"
        f["properties"]["color"] = color_map["No data"]
        f["properties"]["elevation"] = 0

# 4. Render
deck_cons = pdk.Deck(
    layers=[pdk.Layer("GeoJsonLayer", cons_geojson, extruded=True, pickable=True, filled=True, get_elevation="properties.elevation",
                      elevation_scale=10, get_fill_color="properties.color", get_line_color=[255, 255, 255, 100])],
    initial_view_state=pdk.ViewState(latitude=20, longitude=0, zoom=1.5, pitch=40),
    map_provider="mapbox", # <-- FONDAMENTALE
    map_style="mapbox://styles/mapbox/light-v10", 
    api_keys={"mapbox": MAPBOX_TOKEN},
    tooltip={"html": "<b>{name}</b><br/>{category}", "style": {"backgroundColor": "white", "color": "black"}}
)

# Save
deck_cons.to_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\18_global_map_cons.html")
# Show
deck_cons.show()

<div style="
    text-align: center; 
    background: linear-gradient(145deg, #2a2a2a, #1a1a1a); 
    border: 1px solid #444; 
    padding: 30px; 
    border-radius: 16px; 
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    margin: 20px 0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
">
    <div style="color: #ffcc00; font-size: 24px; margin-bottom: 10px;">🔥</div>
    <h3 style="color: #ffffff; margin-top: 0; font-weight: 600;">Security Note: Interactive Consumption Map</h3>
    <p style="color: #cccccc; font-size: 1.1em; line-height: 1.6;">
        Due to security protocols regarding the Mapbox API token, 
        this interactive visualization is hosted externally.
    </p>
    <a href="./Plot/18_global_map_cons.html" target="_blank" style="
        display: inline-block; 
        margin-top: 15px; 
        padding: 12px 25px; 
        background: linear-gradient(to right, #f2994a, #f2c94c); 
        color: white; 
        text-decoration: none; 
        border-radius: 50px; 
        font-weight: bold;
        box-shadow: 0 4px 10px rgba(242, 201, 76, 0.3);
    ">
        🌍 Open Interactive Consumption Map
    </a>
</div>


<div style="text-align:center; border:2px solid #383838; padding:25px; border-radius:12px;">

<span style="font-size:22px; font-weight:700;">
The Illusion of Green Grids: Electricity vs. Total Primary Consumption<br>
</span>

<br>

<span style="font-size:18px;">
This dual 3D visualization highlights the critical discrepancy between a nation's electrical grid (Map 1) and its total primary energy consumption (Map 2).
</span>

<br><br>

<span style="font-size:22px; font-weight:700;">
Methodological Alignment<br>
</span>

<span style="font-size:18px;">
To ensure a rigorous visual comparison, geographical coverage is strictly synchronized. 
Total primary consumption data—which includes heating, industrial manufacturing, and transportation—is often unavailable for smaller economies.<br>
We have therefore filtered Map 1 to include only the countries present in Map 2, ensuring a perfectly consistent "No Data" baseline and preventing visual artifacts between the two views.
</span>

<br><br>

<span style="font-size:22px; font-weight:700;">
Key Analytical Findings
</span>

<br>

<span style="font-size:18px;">
Comparing the two maps reveals what we call the <i>Green Grid Illusion</i>. 
Many developed nations appear highly decarbonized when looking only at their electricity generation (Map 1, shifting toward <b>Gold</b> or <b>Soft Blue</b>).<br>
However, Map 2 exposes the broader economic reality: once heavy industry, transportation, and heating are included, the map shifts dramatically toward <b>Dark Red</b> (“Fossil Giants”).<br>
This demonstrates that while cleaning the power grid is a vital milestone, the true challenge of the global energy transition lies in the deep electrification of the entire real economy.
</span>

</div>
